# 06 — Extra Diagnostics and Final Exports

This notebook packages the final supervisor-requested diagnostics for the Support-Aware Stream Calibration paper.

It consolidates outputs from the ResNet, ViT, and close-baseline workflows, then produces the final diagnostic artifacts used for paper writing:

- paired bootstrap comparisons for Frozen V3/SASC against Source TS, entropy-mean, and Safe V2;
- close-baseline packaging and comparison tables;
- support-region decomposition for inside-support, high-extrapolation, and lower-fallback cases;
- source-temperature NLL curve and flatness summaries;
- entropy confound diagnostics with controls;
- final manifests and export checklists.

The notebook is prepared for repository review. Outputs and execution counts have been cleared. Large cached logits, datasets, and checkpoints are not stored in this repository.


In [ ]:
# Final Protocol Rerun: Calibration Under Distribution Shift

## Protocol note

In [ ]:
This notebook reruns all official calibration methods from scratch under one frozen experimental protocol.

The notebook is designed to ensure that:

- all official outputs are regenerated from raw data, saved splits, and fixed checkpoints;
- comparisons are separated by regime and not mixed into one unfair global leaderboard;
- no result from older stage folders is used as final evidence;
- no official method variant is selected using final benchmark performance;
- all final tables, figures, and audit outputs produced here are the only official outputs for the repaired protocol.

## Regime separation

In [ ]:
The experiments in this notebook are reported under three separate regimes:

- **Regime A — Clean-source post-hoc calibration**  
  Methods use only clean labeled source calibration data.

- **Regime B — Target-unlabeled adaptation**  
  Methods may use unlabeled target data for fitting, but never target labels.

- **Regime C — Source-side robust calibration**  
  Methods use only source-side calibration data together with predefined synthetic perturbations.

## Reproducibility and fairness principles

In [ ]:
This notebook follows the following frozen protocol principles:

- fixed seeds;
- fixed checkpoint selection rule;
- fixed corruption list;
- fixed severity list;
- fixed metric definitions;
- fixed split definitions;
- fixed method configurations decided before final evaluation;
- explicit leakage and fairness audit at the end.

## Official output policy

In [ ]:
All official outputs from this notebook will be saved into a new results directory.  
These outputs include:

- per-seed result files,
- aggregated result files,
- fit-summary files,
- regime-specific comparison tables,
- figures,
- leakage and fairness audit tables,
- protocol metadata.



from pathlib import Path
from datetime import datetime
import json
import pprint

RUN_NAME = "protocol_rerun_v1"
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

PATHS = {
    "drive_root": "/content/drive/MyDrive",
    "project_root": "/content/drive/MyDrive/Advance analysis",
    "notebook_path": "/content/drive/MyDrive/Advance analysis/final.ipynb",

    "imagenet100_zip": "/content/drive/MyDrive/imagenet 100/archive (1).zip",
    "imagenet100_local_root": "/content/imagenet100",

    "imagenetc_tar_root": "/content/drive/MyDrive/imagenet-c",
    "imagenetc_local_root": "/content/imagenet-c",
    "imagenetc_filtered_root": "/content/imagenet-c-100",


    "splits_root": "/content/drive/MyDrive/AML_runs/splits",

    "checkpoints_root": "/content/drive/MyDrive/week 8-10 results ",


    "results_root": f"/content/drive/MyDrive/AML_runs/{RUN_NAME}",
}


PROTOCOL = {
    "run_name": RUN_NAME,
    "run_timestamp": RUN_TIMESTAMP,

    "checkpoint_rule": "best_by_nll.pt",
    "seeds": [1, 2, 3],

    "corruptions": [
        "brightness",
        "defocus_blur",
        "glass_blur",
        "motion_blur",
        "zoom_blur",
        "gaussian_noise",
        "shot_noise",
        "impulse_noise",
        "fog",
        "snow",
    ],

    "severities": [1, 2, 3, 4, 5],

    "metrics": [
        "accuracy",
        "nll",
        "brier",
        "ece10",
        "ece15",
        "ece20",
        "aece15",
    ],

    "regimes": {
        "Regime_A_clean_source_posthoc": [
            "baseline",
            "ts",
            "tsp",
        ],
        "Regime_B_target_unlabeled": [
            "baseline",
            "pseudocal",
        ],
        "Regime_C_source_robust": [
            "baseline",
            "mmnll_ts",
        ],
    },

    "official_output_policy": (
        "All outputs produced by this notebook under the frozen protocol "
        "are the only official outputs for the final rerun."
    ),
}

METHOD_CONFIG = {
    "baseline": {
        "fit_required": False,
        "regime": ["Regime_A_clean_source_posthoc", "Regime_B_target_unlabeled", "Regime_C_source_robust"],
    },

    "ts": {
        "fit_required": True,
        "fit_source": "clean_labeled_calibration_split",
        "temperature_type": "single_scalar",
    },

    "tsp": {
        "fit_required": True,
        "fit_source": "source_calibration_with_fixed_proxy_perturbations",
        "temperature_type": "single_scalar",
        "note": "Exact perturbation rule will be frozen before method execution.",
    },

    "pseudocal": {
        "fit_required": True,
        "fit_source": "disjoint_target_fit_subset_unlabeled",
        "target_eval_policy": "evaluate_only_on_disjoint_target_eval_subset",
        "pseudo_label_threshold": 0.90,
        "temperature_type": "single_scalar_constrained",
    },

    "mmnll_ts": {
        "fit_required": True,
        "fit_source": "source_calibration_with_fixed_perturbations",
        "objective": "min_worst_case_perturbed_nll",
        "temperature_type": "single_scalar_grid_search",
    },
}


RESULT_SUBDIRS = [
    "baseline",
    "ts",
    "tsp",
    "pseudocal",
    "mmnll",
    "aggregated",
    "tables",
    "figures",
    "audit",
    "manifests",
    "metadata",
]

Path(PATHS["results_root"]).mkdir(parents=True, exist_ok=True)
for subdir in RESULT_SUBDIRS:
    Path(PATHS["results_root"], subdir).mkdir(parents=True, exist_ok=True)


protocol_bundle = {
    "PATHS": PATHS,
    "PROTOCOL": PROTOCOL,
    "METHOD_CONFIG": METHOD_CONFIG,
    "RESULT_SUBDIRS": RESULT_SUBDIRS,
}

protocol_json_path = Path(PATHS["results_root"]) / "metadata" / "protocol_summary.json"
with open(protocol_json_path, "w") as f:
    json.dump(protocol_bundle, f, indent=2)

## Imports

In [ ]:
import os
import json
import math
import random
import shutil
import tarfile
import zipfile
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, Dataset

from sklearn.metrics import log_loss


from google.colab import drive
drive.mount('/content/drive')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

if torch.cuda.is_available():
    print("GPU device name:", torch.cuda.get_device_name(0))
    print("CUDA device count:", torch.cuda.device_count())

print("\nKey path existence checks:")
for key, path in PATHS.items():
    exists = Path(path).exists()
    print(f"{key}: {path} | exists={exists}")

OFFICIAL_METHODS = set(METHOD_CONFIG.keys())

# Expected official regimes after removing robust_avg_ts
EXPECTED_REGIMES = {
    "Regime_A_clean_source_posthoc": {"baseline", "ts", "tsp"},
    "Regime_B_target_unlabeled": {"baseline", "pseudocal"},
    "Regime_C_source_robust": {"baseline", "mmnll_ts"},
}

# Validate regime definitions
assert "regimes" in PROTOCOL, "PROTOCOL must contain regime definitions."

for regime_name, expected_methods in EXPECTED_REGIMES.items():
    assert regime_name in PROTOCOL["regimes"], f"Missing regime: {regime_name}"
    found_methods = set(PROTOCOL["regimes"][regime_name])
    assert found_methods == expected_methods, (
        f"Regime mismatch for {regime_name}.\n"
        f"Expected: {expected_methods}\n"
        f"Found: {found_methods}"
    )

all_regime_methods = set()
for regime_methods in PROTOCOL["regimes"].values():
    all_regime_methods.update(regime_methods)

missing_from_config = all_regime_methods - OFFICIAL_METHODS
assert len(missing_from_config) == 0, (
    f"These methods appear in regime definitions but not in METHOD_CONFIG: {missing_from_config}"
)

print("Frozen regime definitions validated successfully.")

assert PROTOCOL["checkpoint_rule"] == "best_by_nll.pt", (
    "Checkpoint rule must remain frozen as best_by_nll.pt"
)

assert PROTOCOL["seeds"] == [1, 2, 3], (
    f"Unexpected seed list: {PROTOCOL['seeds']}"
)

assert PROTOCOL["severities"] == [1, 2, 3, 4, 5], (
    f"Unexpected severity list: {PROTOCOL['severities']}"
)

expected_metrics = ["accuracy", "nll", "brier", "ece10", "ece15", "ece20", "aece15"]
assert PROTOCOL["metrics"] == expected_metrics, (
    f"Unexpected metrics list: {PROTOCOL['metrics']}"
)

expected_corruptions = [
    "brightness",
    "defocus_blur",
    "glass_blur",
    "motion_blur",
    "zoom_blur",
    "gaussian_noise",
    "shot_noise",
    "impulse_noise",
    "fog",
    "snow",
]
assert PROTOCOL["corruptions"] == expected_corruptions, (
    f"Unexpected corruption list: {PROTOCOL['corruptions']}"
)

assert METHOD_CONFIG["pseudocal"]["pseudo_label_threshold"] == 0.90, (
    "PseudoCal threshold must remain frozen at 0.90"
)

print("Frozen protocol fields validated successfully.")

expected_result_subdirs = [
    "baseline",
    "ts",
    "tsp",
    "pseudocal",
    "mmnll",
    "aggregated",
    "tables",
    "figures",
    "audit",
    "manifests",
    "metadata",
]

results_root = Path(PATHS["results_root"])
results_root.mkdir(parents=True, exist_ok=True)

for subdir in expected_result_subdirs:
    (results_root / subdir).mkdir(parents=True, exist_ok=True)



FINAL_RESULT_SUBDIRS = expected_result_subdirs

protocol_bundle = {
    "PATHS": PATHS,
    "PROTOCOL": PROTOCOL,
    "METHOD_CONFIG": METHOD_CONFIG,
    "RESULT_SUBDIRS": FINAL_RESULT_SUBDIRS,
}

protocol_json_path = Path(PATHS["results_root"]) / "metadata" / "protocol_summary.json"
with open(protocol_json_path, "w") as f:
    json.dump(protocol_bundle, f, indent=2)

print("Refreshed protocol summary saved to:")
print(protocol_json_path)


print("OFFICIAL FROZEN EXPERIMENT SUMMARY")


print(f"Run name: {PROTOCOL['run_name']}")
print(f"Run timestamp: {PROTOCOL['run_timestamp']}")
print(f"Checkpoint rule: {PROTOCOL['checkpoint_rule']}")
print(f"Seeds: {PROTOCOL['seeds']}")
print(f"Corruptions: {PROTOCOL['corruptions']}")
print(f"Severities: {PROTOCOL['severities']}")
print(f"Metrics: {PROTOCOL['metrics']}")

print("\nRegimes:")
for regime_name, methods in PROTOCOL["regimes"].items():
    print(f"  {regime_name}: {methods}")

print("\nMethod config summary:")
for method_name, cfg in METHOD_CONFIG.items():
    print(f"\n[{method_name}]")
    for k, v in cfg.items():
        print(f"  {k}: {v}")

## Seeds reproducibility

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # For reproducibility in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Seed control function defined.")

IMAGE_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 2
PIN_MEMORY = torch.cuda.is_available()

RUNTIME_CONFIG = {
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
    "device": str(DEVICE),
}


# Standard ImageNet evaluation transform
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


# For TS-P and MM-NLL sections.
source_base_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
])


runtime_snapshot = {
    "RUNTIME_CONFIG": RUNTIME_CONFIG,
    "eval_transform": {
        "resize": 256,
        "center_crop": IMAGE_SIZE,
        "normalize_mean": [0.485, 0.456, 0.406],
        "normalize_std": [0.229, 0.224, 0.225],
    },
    "notes": (
        "Source-side perturbation transforms are intentionally not frozen here. "
        "They will be defined explicitly inside the TS-P and MM-NLL sections."
    ),
}

runtime_json_path = Path(PATHS["results_root"]) / "metadata" / "runtime_config.json"
with open(runtime_json_path, "w") as f:
    json.dump(runtime_snapshot, f, indent=2)

print("Runtime configuration saved to:")
print(runtime_json_path)

DEFAULT_SEED = PROTOCOL["seeds"][0]
set_seed(DEFAULT_SEED)

## Data extraction

In [ ]:
imagenet100_zip_path = Path(PATHS["imagenet100_zip"])
imagenet100_local_root = Path(PATHS["imagenet100_local_root"])

imagenetc_tar_root = Path(PATHS["imagenetc_tar_root"])
imagenetc_local_root = Path(PATHS["imagenetc_local_root"])


assert imagenet100_zip_path.exists(), f"Missing ImageNet-100 zip file: {imagenet100_zip_path}"

if imagenet100_local_root.exists():
    shutil.rmtree(imagenet100_local_root)

imagenet100_local_root.mkdir(parents=True, exist_ok=True)

print(f"Extracting ImageNet-100 from:\n  {imagenet100_zip_path}")
print(f"to:\n  {imagenet100_local_root}")

with zipfile.ZipFile(imagenet100_zip_path, "r") as zf:
    zf.extractall(imagenet100_local_root)

print("ImageNet-100 extraction completed.")

assert imagenetc_tar_root.exists(), f"Missing ImageNet-C tar root: {imagenetc_tar_root}"
assert imagenetc_tar_root.is_dir(), f"ImageNet-C tar root is not a directory: {imagenetc_tar_root}"

tar_files = sorted(
    [p for p in imagenetc_tar_root.iterdir() if p.is_file() and (p.suffix == ".tar" or p.name.endswith(".tar"))]
)

assert len(tar_files) > 0, f"No .tar files found inside: {imagenetc_tar_root}"

if imagenetc_local_root.exists():
    shutil.rmtree(imagenetc_local_root)

imagenetc_local_root.mkdir(parents=True, exist_ok=True)

print("Found ImageNet-C tar files:")
for tf in tar_files:
    print(" ", tf.name)

for tf_path in tar_files:
    print(f"\nExtracting {tf_path.name} ...")
    with tarfile.open(tf_path, "r") as tf:
        tf.extractall(imagenetc_local_root)

print("\nImageNet-C extraction completed.")

def list_subdirs(path):
    path = Path(path)
    if not path.exists():
        return []
    return sorted([p.name for p in path.iterdir() if p.is_dir()])

top_dirs_100 = list_subdirs(imagenet100_local_root)

candidate_roots = [imagenet100_local_root] + [imagenet100_local_root / d for d in top_dirs_100]

resolved_imagenet100_root = None
for root in candidate_roots:
    if (root / "train").exists() and (root / "val").exists():
        resolved_imagenet100_root = root
        break

assert resolved_imagenet100_root is not None, (
    "Could not find train/ and val/ inside extracted ImageNet-100."
)

CLEAN_TRAIN_DIR = resolved_imagenet100_root / "train"
CLEAN_VAL_DIR = resolved_imagenet100_root / "val"

print("Resolved ImageNet-100 root:", resolved_imagenet100_root)
print("Train dir:", CLEAN_TRAIN_DIR)
print("Val dir  :", CLEAN_VAL_DIR)

train_classes = list_subdirs(CLEAN_TRAIN_DIR)
val_classes = list_subdirs(CLEAN_VAL_DIR)
imagenetc_top_dirs = list_subdirs(imagenetc_local_root)


print("EXTRACTION SUMMARY")

print("ImageNet-100")
print("  train dir exists  :", CLEAN_TRAIN_DIR.exists())
print("  val dir exists    :", CLEAN_VAL_DIR.exists())
print("  train classes     :", len(train_classes))
print("  val classes       :", len(val_classes))
print("  sample classes    :", train_classes[:10])

print("\nImageNet-C")
print("  local root exists :", imagenetc_local_root.exists())
print("  top-level entries :", len(imagenetc_top_dirs))
print("  sample entries    :", imagenetc_top_dirs[:10])

## Fixed class subset, filtered ImageNet-C, and saved splits

In [ ]:
This section prepares the official dataset structure used by all later experiments.

The goals of this section are:

- extract the exact ImageNet-100 class list from the clean dataset,
- filter ImageNet-C so that it contains only the matching 100 classes,
- load or create the saved source split files,
- rebuild the clean source datasets and subsets from the frozen split definition.

This section is critical for reproducibility because all later methods must use the same class subset, the same class ordering, and the same source calibration split.

imagenet100_classes = sorted([p.name for p in CLEAN_TRAIN_DIR.iterdir() if p.is_dir()])

assert len(imagenet100_classes) == 100, (
    f"Expected 100 classes in clean ImageNet-100, but found {len(imagenet100_classes)}"
)

print("Official ImageNet-100 class count:", len(imagenet100_classes))
print("Sample classes:", imagenet100_classes[:10])

imagenetc_filtered_root = Path(PATHS["imagenetc_filtered_root"])

# Remove old filtered copy so this rerun is clean
if imagenetc_filtered_root.exists():
    shutil.rmtree(imagenetc_filtered_root)

imagenetc_filtered_root.mkdir(parents=True, exist_ok=True)

imagenetc_corruptions = sorted([p.name for p in imagenetc_local_root.iterdir() if p.is_dir()])
assert len(imagenetc_corruptions) > 0, "No corruption folders found inside extracted ImageNet-C."

print("Building filtered ImageNet-C-100...")
print("Corruptions found:", imagenetc_corruptions)

for corruption in tqdm(imagenetc_corruptions, desc="Filtering corruptions"):
    corruption_src = imagenetc_local_root / corruption
    corruption_dst = imagenetc_filtered_root / corruption
    corruption_dst.mkdir(parents=True, exist_ok=True)

    for severity in ["1", "2", "3", "4", "5"]:
        severity_src = corruption_src / severity
        severity_dst = corruption_dst / severity
        severity_dst.mkdir(parents=True, exist_ok=True)

        assert severity_src.exists(), f"Missing severity folder: {severity_src}"

        for cls_name in imagenet100_classes:
            src_cls_dir = severity_src / cls_name
            dst_cls_dir = severity_dst / cls_name

            assert src_cls_dir.exists(), (
                f"Missing class {cls_name} under {severity_src}. "
                "This means ImageNet-C class folders do not match ImageNet-100."
            )

            shutil.copytree(src_cls_dir, dst_cls_dir)

print("Filtered ImageNet-C-100 created successfully at:")
print(imagenetc_filtered_root)

filtered_corruptions = sorted([p.name for p in imagenetc_filtered_root.iterdir() if p.is_dir()])

print("Filtered corruptions:", filtered_corruptions)

for corruption in filtered_corruptions[:3]:
    for severity in ["1", "2", "3", "4", "5"]:
        cls_dirs = sorted([p.name for p in (imagenetc_filtered_root / corruption / severity).iterdir() if p.is_dir()])
        assert cls_dirs == imagenet100_classes, (
            f"Filtered class mismatch in {corruption}/{severity}"
        )

print("Filtered ImageNet-C-100 verification passed.")

CLASS_TO_IDX = {cls_name: idx for idx, cls_name in enumerate(imagenet100_classes)}
IDX_TO_CLASS = {idx: cls_name for cls_name, idx in CLASS_TO_IDX.items()}

class_mapping_bundle = {
    "classes": imagenet100_classes,
    "class_to_idx": CLASS_TO_IDX,
    "idx_to_class": IDX_TO_CLASS,
}

class_mapping_path = Path(PATHS["results_root"]) / "metadata" / "class_mapping.json"
with open(class_mapping_path, "w") as f:
    json.dump(class_mapping_bundle, f, indent=2)

print("Class mapping saved to:")
print(class_mapping_path)

full_clean_train_dataset = datasets.ImageFolder(
    root=str(CLEAN_TRAIN_DIR),
    transform=eval_transform
)

full_clean_val_dataset = datasets.ImageFolder(
    root=str(CLEAN_VAL_DIR),
    transform=eval_transform
)

print("Full clean train size:", len(full_clean_train_dataset))
print("Full clean val size  :", len(full_clean_val_dataset))

assert full_clean_train_dataset.classes == imagenet100_classes, (
    "Clean training dataset class order does not match frozen ImageNet-100 class list."
)

assert full_clean_val_dataset.classes == imagenet100_classes, (
    "Clean validation dataset class order does not match frozen ImageNet-100 class list."
)

print("Clean dataset class ordering verified.")

splits_root = Path(PATHS["splits_root"])
splits_root.mkdir(parents=True, exist_ok=True)

train_idx_path = splits_root / "imagenet100_train_idx.json"
calib_idx_path = splits_root / "imagenet100_calib_idx.json"
split_meta_path = splits_root / "imagenet100_split_meta.json"

N = len(full_clean_train_dataset)
all_indices = list(range(N))

if train_idx_path.exists() and calib_idx_path.exists() and split_meta_path.exists():
    with open(train_idx_path, "r") as f:
        train_idx = json.load(f)
    with open(calib_idx_path, "r") as f:
        calib_idx = json.load(f)
    with open(split_meta_path, "r") as f:
        split_meta = json.load(f)

    print("Loaded existing saved split files.")

else:
    print("Saved split files not found. Creating them now under the frozen protocol...")

    set_seed(PROTOCOL["seeds"][0])

    shuffled = all_indices.copy()
    random.shuffle(shuffled)

    calib_fraction = 0.10
    n_calib = int(round(calib_fraction * N))

    calib_idx = sorted(shuffled[:n_calib])
    train_idx = sorted(shuffled[n_calib:])

    split_meta = {
        "split_name": "imagenet100_source_split",
        "dataset_size": N,
        "calib_fraction": calib_fraction,
        "n_train": len(train_idx),
        "n_calib": len(calib_idx),
        "seed_used_for_split_creation": PROTOCOL["seeds"][0],
        "note": (
            "These saved indices define the frozen clean source split reused across all methods."
        ),
    }

    with open(train_idx_path, "w") as f:
        json.dump(train_idx, f)
    with open(calib_idx_path, "w") as f:
        json.dump(calib_idx, f)
    with open(split_meta_path, "w") as f:
        json.dump(split_meta, f, indent=2)

    print("Saved new frozen split files.")

print("Train subset size:", len(train_idx))
print("Calibration subset size:", len(calib_idx))

train_idx_set = set(train_idx)
calib_idx_set = set(calib_idx)

assert len(train_idx) + len(calib_idx) == len(full_clean_train_dataset), (
    "Train + calibration split size does not match full clean training dataset."
)

assert len(train_idx_set.intersection(calib_idx_set)) == 0, (
    "Train and calibration splits overlap."
)

assert train_idx_set.union(calib_idx_set) == set(range(len(full_clean_train_dataset))), (
    "Train and calibration splits do not fully cover the clean training dataset."
)

print("Source split integrity check passed.")

source_train_dataset = Subset(full_clean_train_dataset, train_idx)
source_calib_dataset = Subset(full_clean_train_dataset, calib_idx)
clean_val_dataset = full_clean_val_dataset

print("Official source train subset size :", len(source_train_dataset))
print("Official source calib subset size :", len(source_calib_dataset))
print("Official clean val dataset size   :", len(clean_val_dataset))

source_calib_loader = DataLoader(
    source_calib_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

clean_val_loader = DataLoader(
    clean_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print("Shared clean loaders created successfully.")

section5_summary = {
    "imagenet100_class_count": len(imagenet100_classes),
    "imagenet100_classes_preview": imagenet100_classes[:10],
    "filtered_imagenetc_root": str(imagenetc_filtered_root),
    "filtered_corruptions": filtered_corruptions,
    "source_split_meta": split_meta,
    "source_train_size": len(source_train_dataset),
    "source_calib_size": len(source_calib_dataset),
    "clean_val_size": len(clean_val_dataset),
}

section5_summary_path = Path(PATHS["results_root"]) / "metadata" / "section5_dataset_split_summary.json"
with open(section5_summary_path, "w") as f:
    json.dump(section5_summary, f, indent=2)

print("Section 5 summary saved to:")
print(section5_summary_path)

## Shared metrics and utilities

In [ ]:
This section defines the reusable functions that all later experiment blocks must use.

The goals of this section are:

- define the official metric functions,
- define shared logits extraction utilities,
- define checkpoint loading utilities,
- define ImageNet-C loader construction,
- define shared result-saving helpers.

This section is critical for fairness because every method must use the same evaluation code, the same metric definitions, and the same saving logic.

def softmax_probs_from_logits(logits: torch.Tensor) -> torch.Tensor:
    return torch.softmax(logits, dim=1)

def accuracy_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == labels).float().mean().item()

def nll_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> float:
    return F.cross_entropy(logits, labels, reduction="mean").item()

def brier_from_logits(logits: torch.Tensor, labels: torch.Tensor, num_classes: int = None) -> float:
    probs = softmax_probs_from_logits(logits)
    if num_classes is None:
        num_classes = probs.shape[1]
    one_hot = F.one_hot(labels, num_classes=num_classes).float()
    return torch.mean(torch.sum((probs - one_hot) ** 2, dim=1)).item()

print("Basic metric helpers defined.")

def ece_from_logits(logits: torch.Tensor, labels: torch.Tensor, n_bins: int = 15) -> float:
    probs = softmax_probs_from_logits(logits)
    confidences, predictions = torch.max(probs, dim=1)
    accuracies = predictions.eq(labels)

    bin_boundaries = torch.linspace(0, 1, n_bins + 1, device=logits.device)
    ece = torch.zeros(1, device=logits.device)

    for i in range(n_bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i + 1]

        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = in_bin.float().mean()

        if prop_in_bin.item() > 0:
            acc_in_bin = accuracies[in_bin].float().mean()
            conf_in_bin = confidences[in_bin].mean()
            ece += torch.abs(conf_in_bin - acc_in_bin) * prop_in_bin

    return ece.item()

def adaptive_ece_from_logits(logits: torch.Tensor, labels: torch.Tensor, n_bins: int = 15) -> float:
    probs = softmax_probs_from_logits(logits)
    confidences, predictions = torch.max(probs, dim=1)
    accuracies = predictions.eq(labels).float()

    confidences_sorted, indices = torch.sort(confidences)
    accuracies_sorted = accuracies[indices]

    N = len(confidences_sorted)
    bin_sizes = [N // n_bins] * n_bins
    for i in range(N % n_bins):
        bin_sizes[i] += 1

    aece = 0.0
    start = 0
    for bin_size in bin_sizes:
        end = start + bin_size
        if end <= start:
            continue

        conf_bin = confidences_sorted[start:end]
        acc_bin = accuracies_sorted[start:end]

        if len(conf_bin) > 0:
            conf_mean = conf_bin.mean().item()
            acc_mean = acc_bin.mean().item()
            aece += abs(conf_mean - acc_mean) * (len(conf_bin) / N)

        start = end

    return aece

print("Calibration error metrics defined.")

def metrics_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> dict:
    logits = logits.detach().cpu()
    labels = labels.detach().cpu()

    return {
        "accuracy": accuracy_from_logits(logits, labels),
        "nll": nll_from_logits(logits, labels),
        "brier": brier_from_logits(logits, labels, num_classes=logits.shape[1]),
        "ece10": ece_from_logits(logits, labels, n_bins=10),
        "ece15": ece_from_logits(logits, labels, n_bins=15),
        "ece20": ece_from_logits(logits, labels, n_bins=20),
        "aece15": adaptive_ece_from_logits(logits, labels, n_bins=15),
    }



def apply_temperature(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    assert temperature > 0, f"Temperature must be positive, got {temperature}"
    return logits / temperature

@torch.no_grad()
def get_logits_labels(model: nn.Module, loader: DataLoader, device: torch.device = DEVICE):
    model.eval()

    all_logits = []
    all_labels = []

    for inputs, labels in tqdm(loader, leave=False):
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(inputs)

        all_logits.append(logits.detach().cpu())
        all_labels.append(labels.detach().cpu())

    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return all_logits, all_labels


def resolve_checkpoint_path(seed: int, checkpoint_rule: str = None) -> Path:
    if checkpoint_rule is None:
        checkpoint_rule = PROTOCOL["checkpoint_rule"]

    ckpt_root = Path(PATHS["checkpoints_root"])
    assert ckpt_root.exists(), f"Checkpoint root does not exist: {ckpt_root}"

    # Search recursively for matching checkpoint files
    matches = list(ckpt_root.rglob(checkpoint_rule))
    assert len(matches) > 0, (
        f"No checkpoint named {checkpoint_rule} found under: {ckpt_root}"
    )

    # Keep only paths that appear to belong to the requested seed
    seed_patterns = [
        f"seed_{seed}",
        f"seed{seed}",
        f"/{seed}/",
    ]

    seed_matches = []
    for m in matches:
        m_str = str(m)
        if any(pattern in m_str for pattern in seed_patterns):
            seed_matches.append(m)

    assert len(seed_matches) > 0, (
        f"Found checkpoints named {checkpoint_rule}, but none matched seed {seed}.\n"
        f"Available matches:\n" + "\n".join(str(m) for m in matches)
    )

    # If more than one match exists, prefer the shortest path as the most direct one
    seed_matches = sorted(seed_matches, key=lambda p: len(str(p)))
    chosen = seed_matches[0]

    print(f"[resolve_checkpoint_path] seed={seed} -> {chosen}")
    return chosen


def build_resnet18_model(num_classes: int = 100) -> nn.Module:
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_resnet18_checkpoint(seed: int, device: torch.device = DEVICE) -> nn.Module:
    ckpt_path = resolve_checkpoint_path(seed)
    model = build_resnet18_model(num_classes=len(imagenet100_classes))

    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)

    if isinstance(checkpoint, dict):
        if "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        elif "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        elif "model" in checkpoint:
            state_dict = checkpoint["model"]
        else:

            state_dict = checkpoint
    else:
        state_dict = checkpoint
    cleaned_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith("module."):
            cleaned_state_dict[k[len("module."):]] = v
        else:
            cleaned_state_dict[k] = v

    missing, unexpected = model.load_state_dict(cleaned_state_dict, strict=False)

    print(f"[load_resnet18_checkpoint] Loaded from: {ckpt_path}")
    if len(missing) > 0:
        print("[load_resnet18_checkpoint] Missing keys:")
        print(missing)
    if len(unexpected) > 0:
        print("[load_resnet18_checkpoint] Unexpected keys:")
        print(unexpected)

    model.to(device)
    model.eval()
    return model

print("ResNet-18 builder and checkpoint loader defined.")

def build_imagenetc_loader(corruption: str, severity: int) -> DataLoader:
    assert corruption in filtered_corruptions, (
        f"Unknown corruption: {corruption}. Available: {filtered_corruptions}"
    )
    assert severity in [1, 2, 3, 4, 5], f"Severity must be in [1,2,3,4,5], got {severity}"

    dataset_root = Path(PATHS["imagenetc_filtered_root"]) / corruption / str(severity)
    assert dataset_root.exists(), f"Missing ImageNet-C filtered dataset root: {dataset_root}"

    dataset = datasets.ImageFolder(
        root=str(dataset_root),
        transform=eval_transform
    )

    assert dataset.classes == imagenet100_classes, (
        f"Class ordering mismatch for {corruption}/{severity}"
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    return loader


def metrics_dict_to_long_rows(
    metrics_dict: dict,
    method: str,
    regime: str,
    seed: int,
    split: str,
    corruption: str = None,
    severity: int = None,
    checkpoint_name: str = None,
    fit_source: str = None,
    eval_source: str = None,
):
    rows = []
    for metric_name, metric_value in metrics_dict.items():
        rows.append({
            "method": method,
            "regime": regime,
            "seed": seed,
            "split": split,
            "corruption": corruption,
            "severity": severity,
            "metric": metric_name,
            "value": float(metric_value),
            "checkpoint_name": checkpoint_name,
            "fit_source": fit_source,
            "eval_source": eval_source,
        })
    return rows


def save_dataframe(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

def save_json(data: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)


def bootstrap_mean_ci(values, n_boot: int = 1000, ci: float = 95.0, seed: int = 123):
    values = np.asarray(values, dtype=float)
    assert values.ndim == 1, "bootstrap_mean_ci expects a 1D array."

    rng = np.random.default_rng(seed)
    boot_means = []

    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boot_means.append(np.mean(sample))

    lower = np.percentile(boot_means, (100 - ci) / 2)
    upper = np.percentile(boot_means, 100 - (100 - ci) / 2)

    return {
        "mean": float(np.mean(values)),
        "ci_lower": float(lower),
        "ci_upper": float(upper),
    }


section6_summary = {
    "official_metrics": PROTOCOL["metrics"],
    "checkpoint_rule": PROTOCOL["checkpoint_rule"],
    "num_classes": len(imagenet100_classes),
    "filtered_corruptions": filtered_corruptions,
    "severities": PROTOCOL["severities"],
    "notes": (
        "All later experiment blocks must use the shared functions defined in Section 6."
    ),
}

section6_summary_path = Path(PATHS["results_root"]) / "metadata" / "section6_utilities_summary.json"
save_json(section6_summary, section6_summary_path)

## Leakage audit schema and dry-run sanity check

In [ ]:
This section prepares the audit structure that will later be used to document fairness and possible leakage risks for each method.

It also runs a lightweight dry-run sanity check before any full experiment rerun begins.

The goals of this section are:

- define a standard audit row schema,
- initialize the audit table container,
- test checkpoint loading,
- test clean validation logits extraction,
- test one ImageNet-C corruption loader,
- test metric computation on one small example path.

This section does not produce official experiment results.  
It only verifies that the pipeline is functioning correctly before the full rerun.

AUDIT_COLUMNS = [
    "method",
    "regime",
    "fitting_data_source",
    "labels_used",
    "target_data_used",
    "target_labels_used",
    "synthetic_perturbations_used",
    "benchmark_selection_used",
    "fit_eval_disjoint",
    "leakage_risk",
    "status",
    "notes",
]

audit_rows = []

def make_audit_row(
    method: str,
    regime: str,
    fitting_data_source: str,
    labels_used: bool,
    target_data_used: bool,
    target_labels_used: bool,
    synthetic_perturbations_used: bool,
    benchmark_selection_used: bool,
    fit_eval_disjoint,
    leakage_risk: str,
    status: str,
    notes: str = "",
):
    row = {
        "method": method,
        "regime": regime,
        "fitting_data_source": fitting_data_source,
        "labels_used": labels_used,
        "target_data_used": target_data_used,
        "target_labels_used": target_labels_used,
        "synthetic_perturbations_used": synthetic_perturbations_used,
        "benchmark_selection_used": benchmark_selection_used,
        "fit_eval_disjoint": fit_eval_disjoint,
        "leakage_risk": leakage_risk,
        "status": status,
        "notes": notes,
    }
    return row

print("Audit schema initialized.")
print("Audit columns:")
for col in AUDIT_COLUMNS:
    print(" -", col)

DRY_RUN_SEED = PROTOCOL["seeds"][0]
DRY_RUN_CORRUPTION = PROTOCOL["corruptions"][0]
DRY_RUN_SEVERITY = PROTOCOL["severities"][0]

print("Dry-run seed      :", DRY_RUN_SEED)
print("Dry-run corruption:", DRY_RUN_CORRUPTION)
print("Dry-run severity  :", DRY_RUN_SEVERITY)

set_seed(DRY_RUN_SEED)

dry_model = load_resnet18_checkpoint(DRY_RUN_SEED, device=DEVICE)
print("Checkpoint loaded successfully for dry-run.")
print("Model type:", type(dry_model).__name__)

dry_clean_logits, dry_clean_labels = get_logits_labels(dry_model, clean_val_loader, device=DEVICE)

print("Dry-run clean logits shape:", tuple(dry_clean_logits.shape))
print("Dry-run clean labels shape:", tuple(dry_clean_labels.shape))

assert dry_clean_logits.ndim == 2, "Expected logits tensor of shape [N, C]"
assert dry_clean_labels.ndim == 1, "Expected labels tensor of shape [N]"
assert dry_clean_logits.shape[0] == dry_clean_labels.shape[0], "Mismatch between logits and labels count"
assert dry_clean_logits.shape[1] == len(imagenet100_classes), "Unexpected number of classes in logits"

print("Clean validation dry-run passed.")

dry_clean_metrics = metrics_from_logits(dry_clean_logits, dry_clean_labels)

print("Dry-run clean metrics:")
for k, v in dry_clean_metrics.items():
    print(f"  {k}: {v:.6f}")

dry_corruption_loader = build_imagenetc_loader(
    corruption=DRY_RUN_CORRUPTION,
    severity=DRY_RUN_SEVERITY
)

dry_corr_logits, dry_corr_labels = get_logits_labels(dry_model, dry_corruption_loader, device=DEVICE)

print("Dry-run corruption logits shape:", tuple(dry_corr_logits.shape))
print("Dry-run corruption labels shape:", tuple(dry_corr_labels.shape))

assert dry_corr_logits.ndim == 2, "Expected corruption logits tensor of shape [N, C]"
assert dry_corr_labels.ndim == 1, "Expected corruption labels tensor of shape [N]"
assert dry_corr_logits.shape[0] == dry_corr_labels.shape[0], "Mismatch between corruption logits and labels count"
assert dry_corr_logits.shape[1] == len(imagenet100_classes), "Unexpected number of classes in corruption logits"

print("ImageNet-C dry-run loader passed.")

dry_corr_metrics = metrics_from_logits(dry_corr_logits, dry_corr_labels)

print("Dry-run corruption metrics:")
for k, v in dry_corr_metrics.items():
    print(f"  {k}: {v:.6f}")

dry_temperature = 1.25
dry_scaled_logits = apply_temperature(dry_clean_logits, dry_temperature)
dry_scaled_metrics = metrics_from_logits(dry_scaled_logits, dry_clean_labels)

print(f"Dry-run temperature scaling check with T={dry_temperature}")
for k, v in dry_scaled_metrics.items():
    print(f"  {k}: {v:.6f}")

print("Temperature application helper works.")

dry_run_summary = {
    "seed": DRY_RUN_SEED,
    "corruption": DRY_RUN_CORRUPTION,
    "severity": DRY_RUN_SEVERITY,
    "clean_logits_shape": list(dry_clean_logits.shape),
    "clean_labels_shape": list(dry_clean_labels.shape),
    "corruption_logits_shape": list(dry_corr_logits.shape),
    "corruption_labels_shape": list(dry_corr_labels.shape),
    "clean_metrics": dry_clean_metrics,
    "corruption_metrics": dry_corr_metrics,
    "temperature_test": {
        "temperature": dry_temperature,
        "scaled_metrics": dry_scaled_metrics,
    },
    "status": "passed",
}

dry_run_summary_path = Path(PATHS["results_root"]) / "metadata" / "dry_run_summary.json"
save_json(dry_run_summary, dry_run_summary_path)

print("Dry-run summary saved to:")
print(dry_run_summary_path)

del dry_model
torch.cuda.empty_cache()

print("Dry-run cleanup completed.")

## Baseline rerun

In [ ]:
This section reruns the uncalibrated baseline from scratch under the frozen protocol.

The baseline uses:

- the official checkpoint for each seed,
- no calibration fitting,
- the shared clean validation loader,
- the shared filtered ImageNet-C evaluation loaders.

This baseline serves as the reference method for all regimes.

Outputs from this section include:

- per-seed long-format result files,
- one combined all-seeds result file,
- one aggregated summary file,
- one audit record.

BASELINE_METHOD = "baseline"
BASELINE_REGIME = "reference_all_regimes"
BASELINE_OUTPUT_DIR = Path(PATHS["results_root"]) / "baseline"

BASELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Baseline output dir:", BASELINE_OUTPUT_DIR)
print("Seeds:", PROTOCOL["seeds"])
print("Corruptions:", PROTOCOL["corruptions"])
print("Severities:", PROTOCOL["severities"])

baseline_all_rows = []

for seed in PROTOCOL["seeds"]:
    print("=" * 80)
    print(f"Running baseline for seed {seed}")
    print("=" * 80)

    set_seed(seed)
    model = load_resnet18_checkpoint(seed, device=DEVICE)
    checkpoint_name = PROTOCOL["checkpoint_rule"]

    seed_rows = []

    # Clean validation

    clean_logits, clean_labels = get_logits_labels(model, clean_val_loader, device=DEVICE)
    clean_metrics = metrics_from_logits(clean_logits, clean_labels)

    seed_rows.extend(
        metrics_dict_to_long_rows(
            metrics_dict=clean_metrics,
            method=BASELINE_METHOD,
            regime=BASELINE_REGIME,
            seed=seed,
            split="clean_val",
            corruption=None,
            severity=None,
            checkpoint_name=checkpoint_name,
            fit_source="none",
            eval_source="clean_val",
        )
    )

    print(f"[Seed {seed}] Clean validation done.")

    # ImageNet-C corruptions

    for corruption in PROTOCOL["corruptions"]:
        for severity in PROTOCOL["severities"]:
            corr_loader = build_imagenetc_loader(corruption=corruption, severity=severity)
            corr_logits, corr_labels = get_logits_labels(model, corr_loader, device=DEVICE)
            corr_metrics = metrics_from_logits(corr_logits, corr_labels)

            seed_rows.extend(
                metrics_dict_to_long_rows(
                    metrics_dict=corr_metrics,
                    method=BASELINE_METHOD,
                    regime=BASELINE_REGIME,
                    seed=seed,
                    split="imagenet_c",
                    corruption=corruption,
                    severity=severity,
                    checkpoint_name=checkpoint_name,
                    fit_source="none",
                    eval_source="imagenet_c_filtered",
                )
            )

        print(f"[Seed {seed}] Finished corruption: {corruption}")

    # Save per-seed CSV
    seed_df = pd.DataFrame(seed_rows)
    seed_csv_path = BASELINE_OUTPUT_DIR / f"baseline_seed_{seed}_long.csv"
    save_dataframe(seed_df, seed_csv_path)

    print(f"[Seed {seed}] Saved per-seed results to:")
    print(seed_csv_path)

    baseline_all_rows.extend(seed_rows)

    del model
    torch.cuda.empty_cache()

baseline_all_df = pd.DataFrame(baseline_all_rows)

print("Combined baseline result shape:", baseline_all_df.shape)
print("Columns:", list(baseline_all_df.columns))

baseline_all_csv_path = BASELINE_OUTPUT_DIR / "baseline_all_seeds_long.csv"
save_dataframe(baseline_all_df, baseline_all_csv_path)

print("Saved combined all-seeds baseline results to:")
print(baseline_all_csv_path)

group_cols = ["method", "regime", "split", "corruption", "severity", "metric"]

baseline_agg_rows = []

for group_key, group_df in baseline_all_df.groupby(group_cols, dropna=False):
    values = group_df["value"].values
    agg_stats = bootstrap_mean_ci(values, n_boot=1000, ci=95.0, seed=123)

    row = dict(zip(group_cols, group_key))
    row.update({
        "n_seeds": len(group_df),
        "mean": agg_stats["mean"],
        "ci_lower": agg_stats["ci_lower"],
        "ci_upper": agg_stats["ci_upper"],
    })
    baseline_agg_rows.append(row)

baseline_agg_df = pd.DataFrame(baseline_agg_rows)

print("Aggregated baseline result shape:", baseline_agg_df.shape)
baseline_agg_df.head()

baseline_agg_csv_path = BASELINE_OUTPUT_DIR / "baseline_aggregated.csv"
save_dataframe(baseline_agg_df, baseline_agg_csv_path)

print("Saved aggregated baseline results to:")
print(baseline_agg_csv_path)

baseline_summary = {
    "method": BASELINE_METHOD,
    "regime": BASELINE_REGIME,
    "fit_required": False,
    "fit_source": "none",
    "seeds": PROTOCOL["seeds"],
    "checkpoint_rule": PROTOCOL["checkpoint_rule"],
    "corruptions": PROTOCOL["corruptions"],
    "severities": PROTOCOL["severities"],
    "per_seed_csvs": [
        str(BASELINE_OUTPUT_DIR / f"baseline_seed_{seed}_long.csv")
        for seed in PROTOCOL["seeds"]
    ],
    "combined_csv": str(baseline_all_csv_path),
    "aggregated_csv": str(baseline_agg_csv_path),
}

baseline_summary_path = BASELINE_OUTPUT_DIR / "baseline_summary.json"
save_json(baseline_summary, baseline_summary_path)

print("Saved baseline summary metadata to:")
print(baseline_summary_path)

audit_rows.append(
    make_audit_row(
        method="baseline",
        regime="reference_all_regimes",
        fitting_data_source="none",
        labels_used=False,
        target_data_used=False,
        target_labels_used=False,
        synthetic_perturbations_used=False,
        benchmark_selection_used=False,
        fit_eval_disjoint="not_applicable",
        leakage_risk="none",
        status="passed",
        notes="Uncalibrated checkpoint evaluation only; no fitting performed."
    )
)

print("Baseline audit row added.")
print("Current number of audit rows:", len(audit_rows))

## Temperature Scaling (TS) rerun

In [ ]:
This section reruns standard Temperature Scaling under the frozen protocol.

For each seed, TS uses:

- the official checkpoint,
- the clean labeled source calibration split only,
- one scalar temperature fitted by minimizing negative log-likelihood on the calibration split.

After fitting, the calibrated model is evaluated on:

- clean validation,
- all filtered ImageNet-C corruptions and severities.

Outputs from this section include:

- per-seed fitted temperatures,
- per-seed long-format result files,
- one combined all-seeds result file,
- one aggregated summary file,
- one audit record.

def fit_temperature_scaling(
    logits: torch.Tensor,
    labels: torch.Tensor,
    init_temperature: float = 1.0,
    max_iter: int = 100,
    lr: float = 0.01,
):
    """
    Fit a single scalar temperature by minimizing NLL on labeled calibration logits.
    """
    logits = logits.clone().float()
    labels = labels.clone().long()

    temperature = torch.nn.Parameter(torch.tensor([init_temperature], dtype=torch.float32))
    optimizer = torch.optim.LBFGS([temperature], lr=lr, max_iter=max_iter)

    def closure():
        optimizer.zero_grad()
        t = torch.clamp(temperature, min=1e-6)
        loss = F.cross_entropy(logits / t, labels)
        loss.backward()
        return loss

    optimizer.step(closure)

    fitted_t = float(torch.clamp(temperature.detach(), min=1e-6).item())
    final_nll = float(F.cross_entropy(logits / fitted_t, labels).item())

    return {
        "temperature": fitted_t,
        "final_calib_nll": final_nll,
        "num_fit_samples": int(len(labels)),
    }

print("TS fitting helper defined.")

TS_METHOD = "ts"
TS_REGIME = "Regime_A_clean_source_posthoc"
TS_OUTPUT_DIR = Path(PATHS["results_root"]) / "ts"

TS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TS output dir:", TS_OUTPUT_DIR)
print("TS regime:", TS_REGIME)

ts_all_rows = []
ts_fit_rows = []

for seed in PROTOCOL["seeds"]:
    print("=" * 80)
    print(f"Running TS for seed {seed}")
    print("=" * 80)

    set_seed(seed)
    model = load_resnet18_checkpoint(seed, device=DEVICE)
    checkpoint_name = PROTOCOL["checkpoint_rule"]


    # Fit on clean labeled calibration split

    calib_logits, calib_labels = get_logits_labels(model, source_calib_loader, device=DEVICE)

    ts_fit = fit_temperature_scaling(calib_logits, calib_labels)
    fitted_T = ts_fit["temperature"]

    ts_fit_rows.append({
        "method": TS_METHOD,
        "regime": TS_REGIME,
        "seed": seed,
        "temperature": fitted_T,
        "final_calib_nll": ts_fit["final_calib_nll"],
        "num_fit_samples": ts_fit["num_fit_samples"],
        "fit_source": "clean_labeled_calibration_split",
        "checkpoint_name": checkpoint_name,
    })

    print(f"[Seed {seed}] Fitted temperature: {fitted_T:.6f}")

    seed_rows = []


    # Clean validation evaluation

    clean_logits, clean_labels = get_logits_labels(model, clean_val_loader, device=DEVICE)
    clean_logits_ts = apply_temperature(clean_logits, fitted_T)
    clean_metrics_ts = metrics_from_logits(clean_logits_ts, clean_labels)

    seed_rows.extend(
        metrics_dict_to_long_rows(
            metrics_dict=clean_metrics_ts,
            method=TS_METHOD,
            regime=TS_REGIME,
            seed=seed,
            split="clean_val",
            corruption=None,
            severity=None,
            checkpoint_name=checkpoint_name,
            fit_source="clean_labeled_calibration_split",
            eval_source="clean_val",
        )
    )

    print(f"[Seed {seed}] Clean validation done.")


    # ImageNet-C evaluation

    for corruption in PROTOCOL["corruptions"]:
        for severity in PROTOCOL["severities"]:
            corr_loader = build_imagenetc_loader(corruption=corruption, severity=severity)
            corr_logits, corr_labels = get_logits_labels(model, corr_loader, device=DEVICE)
            corr_logits_ts = apply_temperature(corr_logits, fitted_T)
            corr_metrics_ts = metrics_from_logits(corr_logits_ts, corr_labels)

            seed_rows.extend(
                metrics_dict_to_long_rows(
                    metrics_dict=corr_metrics_ts,
                    method=TS_METHOD,
                    regime=TS_REGIME,
                    seed=seed,
                    split="imagenet_c",
                    corruption=corruption,
                    severity=severity,
                    checkpoint_name=checkpoint_name,
                    fit_source="clean_labeled_calibration_split",
                    eval_source="imagenet_c_filtered",
                )
            )

        print(f"[Seed {seed}] Finished corruption: {corruption}")

    # Save per-seed long results
    seed_df = pd.DataFrame(seed_rows)
    seed_csv_path = TS_OUTPUT_DIR / f"ts_seed_{seed}_long.csv"
    save_dataframe(seed_df, seed_csv_path)

    print(f"[Seed {seed}] Saved per-seed TS results to:")
    print(seed_csv_path)

    ts_all_rows.extend(seed_rows)

    del model
    torch.cuda.empty_cache()

ts_fit_df = pd.DataFrame(ts_fit_rows)
ts_fit_csv_path = TS_OUTPUT_DIR / "ts_fit_summary.csv"
save_dataframe(ts_fit_df, ts_fit_csv_path)

print("Saved TS fit summary to:")
print(ts_fit_csv_path)

display(ts_fit_df)

ts_all_df = pd.DataFrame(ts_all_rows)

print("Combined TS result shape:", ts_all_df.shape)
print("Columns:", list(ts_all_df.columns))

ts_all_csv_path = TS_OUTPUT_DIR / "ts_all_seeds_long.csv"
save_dataframe(ts_all_df, ts_all_csv_path)

print("Saved combined all-seeds TS results to:")
print(ts_all_csv_path)

group_cols = ["method", "regime", "split", "corruption", "severity", "metric"]

ts_agg_rows = []

for group_key, group_df in ts_all_df.groupby(group_cols, dropna=False):
    values = group_df["value"].values
    agg_stats = bootstrap_mean_ci(values, n_boot=1000, ci=95.0, seed=123)

    row = dict(zip(group_cols, group_key))
    row.update({
        "n_seeds": len(group_df),
        "mean": agg_stats["mean"],
        "ci_lower": agg_stats["ci_lower"],
        "ci_upper": agg_stats["ci_upper"],
    })
    ts_agg_rows.append(row)

ts_agg_df = pd.DataFrame(ts_agg_rows)

print("Aggregated TS result shape:", ts_agg_df.shape)
ts_agg_df.head()

ts_agg_csv_path = TS_OUTPUT_DIR / "ts_aggregated.csv"
save_dataframe(ts_agg_df, ts_agg_csv_path)

print("Saved aggregated TS results to:")
print(ts_agg_csv_path)

ts_summary = {
    "method": TS_METHOD,
    "regime": TS_REGIME,
    "fit_required": True,
    "fit_source": "clean_labeled_calibration_split",
    "seeds": PROTOCOL["seeds"],
    "checkpoint_rule": PROTOCOL["checkpoint_rule"],
    "corruptions": PROTOCOL["corruptions"],
    "severities": PROTOCOL["severities"],
    "fit_summary_csv": str(ts_fit_csv_path),
    "per_seed_csvs": [
        str(TS_OUTPUT_DIR / f"ts_seed_{seed}_long.csv")
        for seed in PROTOCOL["seeds"]
    ],
    "combined_csv": str(ts_all_csv_path),
    "aggregated_csv": str(ts_agg_csv_path),
}

ts_summary_path = TS_OUTPUT_DIR / "ts_summary.json"
save_json(ts_summary, ts_summary_path)

print("Saved TS summary metadata to:")
print(ts_summary_path)

audit_rows.append(
    make_audit_row(
        method="ts",
        regime="Regime_A_clean_source_posthoc",
        fitting_data_source="clean_labeled_calibration_split",
        labels_used=True,
        target_data_used=False,
        target_labels_used=False,
        synthetic_perturbations_used=False,
        benchmark_selection_used=False,
        fit_eval_disjoint=True,
        leakage_risk="low",
        status="passed",
        notes="Standard temperature scaling fit on clean labeled source calibration split only."
    )
)

print("TS audit row added.")
print("Current number of audit rows:", len(audit_rows))

##  TS-P rerun

In [ ]:
This section reruns TS-P under the frozen protocol.

TS-P is a source-side proxy-perturbation variant of temperature scaling.

For each seed, TS-P uses:

- the official checkpoint,
- the clean labeled source calibration split,
- a fixed source-side proxy perturbation rule applied to calibration inputs,
- one scalar temperature fitted by minimizing negative log-likelihood on the perturbed labeled calibration split.

After fitting, the calibrated model is evaluated on:

- clean validation,
- all filtered ImageNet-C corruptions and severities.

This section keeps the perturbation rule explicit and frozen so that TS-P remains scientifically interpretable and reproducible.

TSP_METHOD = "tsp"
TSP_REGIME = "Regime_A_clean_source_posthoc"
TSP_OUTPUT_DIR = Path(PATHS["results_root"]) / "tsp"
TSP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Frozen proxy perturbation configuration
TSP_CONFIG = {
    "resize": 256,
    "center_crop": IMAGE_SIZE,
    "gaussian_blur_kernel_size": 5,
    "gaussian_blur_sigma": 1.0,
    "jitter_brightness": 0.15,
    "jitter_contrast": 0.15,
    "jitter_saturation": 0.05,
    "jitter_hue": 0.02,
}

tsp_proxy_transform = transforms.Compose([
    transforms.Resize(TSP_CONFIG["resize"]),
    transforms.CenterCrop(TSP_CONFIG["center_crop"]),
    transforms.GaussianBlur(
        kernel_size=TSP_CONFIG["gaussian_blur_kernel_size"],
        sigma=TSP_CONFIG["gaussian_blur_sigma"]
    ),
    transforms.ColorJitter(
        brightness=TSP_CONFIG["jitter_brightness"],
        contrast=TSP_CONFIG["jitter_contrast"],
        saturation=TSP_CONFIG["jitter_saturation"],
        hue=TSP_CONFIG["jitter_hue"],
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

print("TS-P output dir:", TSP_OUTPUT_DIR)
print("Frozen TS-P config:")
for k, v in TSP_CONFIG.items():
    print(f"  {k}: {v}")


# Build perturbed source calibration dataset

perturbed_full_clean_train_dataset = datasets.ImageFolder(
    root=str(CLEAN_TRAIN_DIR),
    transform=tsp_proxy_transform
)

assert perturbed_full_clean_train_dataset.classes == imagenet100_classes, (
    "Perturbed clean training dataset class order does not match frozen ImageNet-100 class list."
)

source_calib_dataset_tsp = Subset(perturbed_full_clean_train_dataset, calib_idx)

source_calib_loader_tsp = DataLoader(
    source_calib_dataset_tsp,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print("Perturbed source calibration dataset size:", len(source_calib_dataset_tsp))

tsp_protocol_summary = {
    "method": TSP_METHOD,
    "regime": TSP_REGIME,
    "fit_source": "source_calibration_with_fixed_proxy_perturbations",
    "tsp_config": TSP_CONFIG,
    "note": (
        "TS-P uses one fixed source-side proxy perturbation transform and then fits "
        "a single temperature by standard NLL minimization on the perturbed labeled calibration split."
    ),
}

tsp_protocol_path = TSP_OUTPUT_DIR / "tsp_protocol_summary.json"
save_json(tsp_protocol_summary, tsp_protocol_path)

print("Saved TS-P protocol summary to:")
print(tsp_protocol_path)

tsp_all_rows = []
tsp_fit_rows = []

for seed in PROTOCOL["seeds"]:
    print("=" * 80)
    print(f"Running TS-P for seed {seed}")
    print("=" * 80)

    set_seed(seed)
    model = load_resnet18_checkpoint(seed, device=DEVICE)
    checkpoint_name = PROTOCOL["checkpoint_rule"]


    # Fit on proxy-perturbed labeled source calibration split

    calib_logits_tsp, calib_labels_tsp = get_logits_labels(model, source_calib_loader_tsp, device=DEVICE)

    tsp_fit = fit_temperature_scaling(calib_logits_tsp, calib_labels_tsp)
    fitted_T = tsp_fit["temperature"]

    tsp_fit_rows.append({
        "method": TSP_METHOD,
        "regime": TSP_REGIME,
        "seed": seed,
        "temperature": fitted_T,
        "final_calib_nll": tsp_fit["final_calib_nll"],
        "num_fit_samples": tsp_fit["num_fit_samples"],
        "fit_source": "source_calibration_with_fixed_proxy_perturbations",
        "checkpoint_name": checkpoint_name,
    })

    print(f"[Seed {seed}] Fitted TS-P temperature: {fitted_T:.6f}")

    seed_rows = []


    # Clean validation evaluation

    clean_logits, clean_labels = get_logits_labels(model, clean_val_loader, device=DEVICE)
    clean_logits_tsp = apply_temperature(clean_logits, fitted_T)
    clean_metrics_tsp = metrics_from_logits(clean_logits_tsp, clean_labels)

    seed_rows.extend(
        metrics_dict_to_long_rows(
            metrics_dict=clean_metrics_tsp,
            method=TSP_METHOD,
            regime=TSP_REGIME,
            seed=seed,
            split="clean_val",
            corruption=None,
            severity=None,
            checkpoint_name=checkpoint_name,
            fit_source="source_calibration_with_fixed_proxy_perturbations",
            eval_source="clean_val",
        )
    )

    print(f"[Seed {seed}] Clean validation done.")


    # ImageNet-C evaluation

    for corruption in PROTOCOL["corruptions"]:
        for severity in PROTOCOL["severities"]:
            corr_loader = build_imagenetc_loader(corruption=corruption, severity=severity)
            corr_logits, corr_labels = get_logits_labels(model, corr_loader, device=DEVICE)
            corr_logits_tsp = apply_temperature(corr_logits, fitted_T)
            corr_metrics_tsp = metrics_from_logits(corr_logits_tsp, corr_labels)

            seed_rows.extend(
                metrics_dict_to_long_rows(
                    metrics_dict=corr_metrics_tsp,
                    method=TSP_METHOD,
                    regime=TSP_REGIME,
                    seed=seed,
                    split="imagenet_c",
                    corruption=corruption,
                    severity=severity,
                    checkpoint_name=checkpoint_name,
                    fit_source="source_calibration_with_fixed_proxy_perturbations",
                    eval_source="imagenet_c_filtered",
                )
            )

        print(f"[Seed {seed}] Finished corruption: {corruption}")

    # Save per-seed long results
    seed_df = pd.DataFrame(seed_rows)
    seed_csv_path = TSP_OUTPUT_DIR / f"tsp_seed_{seed}_long.csv"
    save_dataframe(seed_df, seed_csv_path)

    print(f"[Seed {seed}] Saved per-seed TS-P results to:")
    print(seed_csv_path)

    tsp_all_rows.extend(seed_rows)

    del model
    torch.cuda.empty_cache()

tsp_fit_df = pd.DataFrame(tsp_fit_rows)
tsp_fit_csv_path = TSP_OUTPUT_DIR / "tsp_fit_summary.csv"
save_dataframe(tsp_fit_df, tsp_fit_csv_path)

print("Saved TS-P fit summary to:")
print(tsp_fit_csv_path)

display(tsp_fit_df)

tsp_all_df = pd.DataFrame(tsp_all_rows)

print("Combined TS-P result shape:", tsp_all_df.shape)
print("Columns:", list(tsp_all_df.columns))

tsp_all_csv_path = TSP_OUTPUT_DIR / "tsp_all_seeds_long.csv"
save_dataframe(tsp_all_df, tsp_all_csv_path)

print("Saved combined all-seeds TS-P results to:")
print(tsp_all_csv_path)

group_cols = ["method", "regime", "split", "corruption", "severity", "metric"]

tsp_agg_rows = []

for group_key, group_df in tsp_all_df.groupby(group_cols, dropna=False):
    values = group_df["value"].values
    agg_stats = bootstrap_mean_ci(values, n_boot=1000, ci=95.0, seed=123)

    row = dict(zip(group_cols, group_key))
    row.update({
        "n_seeds": len(group_df),
        "mean": agg_stats["mean"],
        "ci_lower": agg_stats["ci_lower"],
        "ci_upper": agg_stats["ci_upper"],
    })
    tsp_agg_rows.append(row)

tsp_agg_df = pd.DataFrame(tsp_agg_rows)

print("Aggregated TS-P result shape:", tsp_agg_df.shape)
tsp_agg_df.head()

tsp_agg_csv_path = TSP_OUTPUT_DIR / "tsp_aggregated.csv"
save_dataframe(tsp_agg_df, tsp_agg_csv_path)

print("Saved aggregated TS-P results to:")
print(tsp_agg_csv_path)

tsp_summary = {
    "method": TSP_METHOD,
    "regime": TSP_REGIME,
    "fit_required": True,
    "fit_source": "source_calibration_with_fixed_proxy_perturbations",
    "seeds": PROTOCOL["seeds"],
    "checkpoint_rule": PROTOCOL["checkpoint_rule"],
    "corruptions": PROTOCOL["corruptions"],
    "severities": PROTOCOL["severities"],
    "tsp_protocol_summary": str(tsp_protocol_path),
    "fit_summary_csv": str(tsp_fit_csv_path),
    "per_seed_csvs": [
        str(TSP_OUTPUT_DIR / f"tsp_seed_{seed}_long.csv")
        for seed in PROTOCOL["seeds"]
    ],
    "combined_csv": str(tsp_all_csv_path),
    "aggregated_csv": str(tsp_agg_csv_path),
}

tsp_summary_path = TSP_OUTPUT_DIR / "tsp_summary.json"
save_json(tsp_summary, tsp_summary_path)

print("Saved TS-P summary metadata to:")
print(tsp_summary_path)

audit_rows.append(
    make_audit_row(
        method="tsp",
        regime="Regime_A_clean_source_posthoc",
        fitting_data_source="source_calibration_with_fixed_proxy_perturbations",
        labels_used=True,
        target_data_used=False,
        target_labels_used=False,
        synthetic_perturbations_used=True,
        benchmark_selection_used=False,
        fit_eval_disjoint=True,
        leakage_risk="low",
        status="passed",
        notes="Single-temperature fit on labeled source calibration data after one fixed proxy perturbation transform."
    )
)

print("TS-P audit row added.")
print("Current number of audit rows:", len(audit_rows))

##  PseudoCal rerun

In [ ]:
This section reruns PseudoCal under the frozen target-unlabeled protocol.

PseudoCal belongs to Regime B and uses:

- the official checkpoint,
- unlabeled target data from a disjoint target-fit subset,
- a fixed pseudo-label confidence threshold,
- one scalar temperature fitted using accepted pseudo-labeled target-fit samples only.

PseudoCal does not use target labels during fitting.

To avoid leakage, each target corruption/severity dataset is split into two disjoint parts:

- **target-fit subset**: used only for pseudo-label fitting,
- **target-eval subset**: used only for final evaluation.

The final benchmark results for PseudoCal are reported only on the disjoint target-eval subset.

PSEUDOCAL_METHOD = "pseudocal"
PSEUDOCAL_REGIME = "Regime_B_target_unlabeled"
PSEUDOCAL_OUTPUT_DIR = Path(PATHS["results_root"]) / "pseudocal"
PSEUDOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PSEUDOCAL_CONFIG = {
    "pseudo_label_threshold": 0.90,
    "target_split_rule": "per corruption/severity dataset, first 50% = fit, second 50% = eval",
    "temperature_bounds": [0.05, 5.0],
    "fit_requires_at_least_one_accepted_sample": True,
}

print("PseudoCal output dir:", PSEUDOCAL_OUTPUT_DIR)
print("Frozen PseudoCal config:")
for k, v in PSEUDOCAL_CONFIG.items():
    print(f"  {k}: {v}")


# Target split helper for PseudoCal

def build_imagenetc_dataset(corruption: str, severity: int):
    assert corruption in filtered_corruptions, (
        f"Unknown corruption: {corruption}. Available: {filtered_corruptions}"
    )
    assert severity in [1, 2, 3, 4, 5], f"Severity must be in [1,2,3,4,5], got {severity}"

    dataset_root = Path(PATHS["imagenetc_filtered_root"]) / corruption / str(severity)
    assert dataset_root.exists(), f"Missing ImageNet-C filtered dataset root: {dataset_root}"

    dataset = datasets.ImageFolder(
        root=str(dataset_root),
        transform=eval_transform
    )

    assert dataset.classes == imagenet100_classes, (
        f"Class ordering mismatch for {corruption}/{severity}"
    )
    return dataset

def build_pseudocal_fit_eval_subsets(corruption: str, severity: int):
    dataset = build_imagenetc_dataset(corruption, severity)
    n = len(dataset)
    assert n >= 2, f"Dataset too small for fit/eval split: {corruption}/{severity}"

    indices = list(range(n))
    midpoint = n // 2

    fit_idx = indices[:midpoint]
    eval_idx = indices[midpoint:]

    assert len(set(fit_idx).intersection(set(eval_idx))) == 0, (
        f"Overlap detected in PseudoCal fit/eval split for {corruption}/{severity}"
    )
    assert len(fit_idx) + len(eval_idx) == n, (
        f"Incomplete fit/eval split for {corruption}/{severity}"
    )

    fit_subset = Subset(dataset, fit_idx)
    eval_subset = Subset(dataset, eval_idx)

    return fit_subset, eval_subset, fit_idx, eval_idx, n

print("PseudoCal target fit/eval split helpers defined.")

def fit_pseudocal_temperature(
    logits: torch.Tensor,
    threshold: float = 0.90,
    t_min: float = 0.05,
    t_max: float = 5.0,
    max_iter: int = 100,
    lr: float = 0.01,
):
    """
    Fit a scalar temperature using pseudo-labels created from model confidence.
    Only samples with max softmax confidence >= threshold are used.
    """
    logits = logits.clone().float()

    with torch.no_grad():
        probs = torch.softmax(logits, dim=1)
        confidences, pseudo_labels = torch.max(probs, dim=1)
        accept_mask = confidences >= threshold

    accepted_logits = logits[accept_mask]
    accepted_pseudo_labels = pseudo_labels[accept_mask]

    num_total = int(len(logits))
    num_accepted = int(len(accepted_logits))
    acceptance_rate = float(num_accepted / max(num_total, 1))

    if num_accepted == 0:
        return {
            "temperature": 1.0,
            "num_total_samples": num_total,
            "num_accepted_samples": 0,
            "acceptance_rate": acceptance_rate,
            "final_fit_nll": None,
            "status": "no_accepted_samples_identity_returned",
        }

    temperature = torch.nn.Parameter(torch.tensor([1.0], dtype=torch.float32))
    optimizer = torch.optim.LBFGS([temperature], lr=lr, max_iter=max_iter)

    def closure():
        optimizer.zero_grad()
        t = torch.clamp(temperature, min=t_min, max=t_max)
        loss = F.cross_entropy(accepted_logits / t, accepted_pseudo_labels)
        loss.backward()
        return loss

    optimizer.step(closure)

    fitted_t = float(torch.clamp(temperature.detach(), min=t_min, max=t_max).item())
    final_fit_nll = float(F.cross_entropy(accepted_logits / fitted_t, accepted_pseudo_labels).item())

    return {
        "temperature": fitted_t,
        "num_total_samples": num_total,
        "num_accepted_samples": num_accepted,
        "acceptance_rate": acceptance_rate,
        "final_fit_nll": final_fit_nll,
        "status": "fit_completed",
    }

print("PseudoCal fitting helper defined.")

pseudocal_protocol_summary = {
    "method": PSEUDOCAL_METHOD,
    "regime": PSEUDOCAL_REGIME,
    "fit_source": "disjoint_target_fit_subset_unlabeled",
    "eval_source": "disjoint_target_eval_subset",
    "config": PSEUDOCAL_CONFIG,
    "note": (
        "PseudoCal uses unlabeled target-fit subsets only for fitting. "
        "Target labels are never used during fitting. Final evaluation is reported "
        "only on disjoint target-eval subsets."
    ),
}

pseudocal_protocol_path = PSEUDOCAL_OUTPUT_DIR / "pseudocal_protocol_summary.json"
save_json(pseudocal_protocol_summary, pseudocal_protocol_path)

print("Saved PseudoCal protocol summary to:")
print(pseudocal_protocol_path)

pseudocal_all_rows = []
pseudocal_fit_rows = []
pseudocal_split_rows = []

for seed in PROTOCOL["seeds"]:
    print("=" * 80)
    print(f"Running PseudoCal for seed {seed}")
    print("=" * 80)

    set_seed(seed)
    model = load_resnet18_checkpoint(seed, device=DEVICE)
    checkpoint_name = PROTOCOL["checkpoint_rule"]

    seed_rows = []

    clean_logits, clean_labels = get_logits_labels(model, clean_val_loader, device=DEVICE)
    clean_metrics_identity = metrics_from_logits(clean_logits, clean_labels)

    seed_rows.extend(
        metrics_dict_to_long_rows(
            metrics_dict=clean_metrics_identity,
            method=PSEUDOCAL_METHOD,
            regime=PSEUDOCAL_REGIME,
            seed=seed,
            split="clean_val",
            corruption=None,
            severity=None,
            checkpoint_name=checkpoint_name,
            fit_source="disjoint_target_fit_subset_unlabeled",
            eval_source="clean_val",
        )
    )

    print(f"[Seed {seed}] Clean validation reference done.")

    for corruption in PROTOCOL["corruptions"]:
        for severity in PROTOCOL["severities"]:
            fit_subset, eval_subset, fit_idx, eval_idx, total_n = build_pseudocal_fit_eval_subsets(
                corruption=corruption,
                severity=severity,
            )

            fit_loader = DataLoader(
                fit_subset,
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
                pin_memory=PIN_MEMORY,
            )

            eval_loader = DataLoader(
                eval_subset,
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
                pin_memory=PIN_MEMORY,
            )

            # Fit uses logits only; labels are returned by loader but ignored in fitting.
            fit_logits, _fit_labels_unused = get_logits_labels(model, fit_loader, device=DEVICE)

            fit_result = fit_pseudocal_temperature(
                logits=fit_logits,
                threshold=PSEUDOCAL_CONFIG["pseudo_label_threshold"],
                t_min=PSEUDOCAL_CONFIG["temperature_bounds"][0],
                t_max=PSEUDOCAL_CONFIG["temperature_bounds"][1],
            )

            fitted_T = fit_result["temperature"]

            pseudocal_fit_rows.append({
                "method": PSEUDOCAL_METHOD,
                "regime": PSEUDOCAL_REGIME,
                "seed": seed,
                "corruption": corruption,
                "severity": severity,
                "temperature": fitted_T,
                "num_total_samples": fit_result["num_total_samples"],
                "num_accepted_samples": fit_result["num_accepted_samples"],
                "acceptance_rate": fit_result["acceptance_rate"],
                "final_fit_nll": fit_result["final_fit_nll"],
                "status": fit_result["status"],
                "fit_source": "disjoint_target_fit_subset_unlabeled",
                "checkpoint_name": checkpoint_name,
            })

            pseudocal_split_rows.append({
                "seed": seed,
                "corruption": corruption,
                "severity": severity,
                "total_samples": total_n,
                "fit_size": len(fit_idx),
                "eval_size": len(eval_idx),
                "fit_eval_overlap": len(set(fit_idx).intersection(set(eval_idx))),
            })

            eval_logits, eval_labels = get_logits_labels(model, eval_loader, device=DEVICE)
            eval_logits_pc = apply_temperature(eval_logits, fitted_T)
            eval_metrics_pc = metrics_from_logits(eval_logits_pc, eval_labels)

            seed_rows.extend(
                metrics_dict_to_long_rows(
                    metrics_dict=eval_metrics_pc,
                    method=PSEUDOCAL_METHOD,
                    regime=PSEUDOCAL_REGIME,
                    seed=seed,
                    split="imagenet_c_target_eval",
                    corruption=corruption,
                    severity=severity,
                    checkpoint_name=checkpoint_name,
                    fit_source="disjoint_target_fit_subset_unlabeled",
                    eval_source="disjoint_target_eval_subset",
                )
            )

        print(f"[Seed {seed}] Finished corruption: {corruption}")

    seed_df = pd.DataFrame(seed_rows)
    seed_csv_path = PSEUDOCAL_OUTPUT_DIR / f"pseudocal_seed_{seed}_long.csv"
    save_dataframe(seed_df, seed_csv_path)

    print(f"[Seed {seed}] Saved per-seed PseudoCal results to:")
    print(seed_csv_path)

    pseudocal_all_rows.extend(seed_rows)

    del model
    torch.cuda.empty_cache()

pseudocal_fit_df = pd.DataFrame(pseudocal_fit_rows)
pseudocal_split_df = pd.DataFrame(pseudocal_split_rows)

pseudocal_fit_csv_path = PSEUDOCAL_OUTPUT_DIR / "pseudocal_fit_summary.csv"
pseudocal_split_csv_path = PSEUDOCAL_OUTPUT_DIR / "pseudocal_split_summary.csv"

save_dataframe(pseudocal_fit_df, pseudocal_fit_csv_path)
save_dataframe(pseudocal_split_df, pseudocal_split_csv_path)

print("Saved PseudoCal fit summary to:")
print(pseudocal_fit_csv_path)
print("Saved PseudoCal split summary to:")
print(pseudocal_split_csv_path)

display(pseudocal_fit_df.head())
display(pseudocal_split_df.head())

pseudocal_all_df = pd.DataFrame(pseudocal_all_rows)

print("Combined PseudoCal result shape:", pseudocal_all_df.shape)
print("Columns:", list(pseudocal_all_df.columns))

pseudocal_all_csv_path = PSEUDOCAL_OUTPUT_DIR / "pseudocal_all_seeds_long.csv"
save_dataframe(pseudocal_all_df, pseudocal_all_csv_path)

print("Saved combined all-seeds PseudoCal results to:")
print(pseudocal_all_csv_path)

group_cols = ["method", "regime", "split", "corruption", "severity", "metric"]

pseudocal_agg_rows = []

for group_key, group_df in pseudocal_all_df.groupby(group_cols, dropna=False):
    values = group_df["value"].values
    agg_stats = bootstrap_mean_ci(values, n_boot=1000, ci=95.0, seed=123)

    row = dict(zip(group_cols, group_key))
    row.update({
        "n_seeds": len(group_df),
        "mean": agg_stats["mean"],
        "ci_lower": agg_stats["ci_lower"],
        "ci_upper": agg_stats["ci_upper"],
    })
    pseudocal_agg_rows.append(row)

pseudocal_agg_df = pd.DataFrame(pseudocal_agg_rows)

print("Aggregated PseudoCal result shape:", pseudocal_agg_df.shape)
pseudocal_agg_df.head()

pseudocal_agg_csv_path = PSEUDOCAL_OUTPUT_DIR / "pseudocal_aggregated.csv"
save_dataframe(pseudocal_agg_df, pseudocal_agg_csv_path)

print("Saved aggregated PseudoCal results to:")
print(pseudocal_agg_csv_path)

pseudocal_summary = {
    "method": PSEUDOCAL_METHOD,
    "regime": PSEUDOCAL_REGIME,
    "fit_required": True,
    "fit_source": "disjoint_target_fit_subset_unlabeled",
    "eval_source": "disjoint_target_eval_subset",
    "threshold": PSEUDOCAL_CONFIG["pseudo_label_threshold"],
    "seeds": PROTOCOL["seeds"],
    "checkpoint_rule": PROTOCOL["checkpoint_rule"],
    "corruptions": PROTOCOL["corruptions"],
    "severities": PROTOCOL["severities"],
    "protocol_summary_json": str(pseudocal_protocol_path),
    "fit_summary_csv": str(pseudocal_fit_csv_path),
    "split_summary_csv": str(pseudocal_split_csv_path),
    "per_seed_csvs": [
        str(PSEUDOCAL_OUTPUT_DIR / f"pseudocal_seed_{seed}_long.csv")
        for seed in PROTOCOL["seeds"]
    ],
    "combined_csv": str(pseudocal_all_csv_path),
    "aggregated_csv": str(pseudocal_agg_csv_path),
}

pseudocal_summary_path = PSEUDOCAL_OUTPUT_DIR / "pseudocal_summary.json"
save_json(pseudocal_summary, pseudocal_summary_path)

print("Saved PseudoCal summary metadata to:")
print(pseudocal_summary_path)

audit_rows.append(
    make_audit_row(
        method="pseudocal",
        regime="Regime_B_target_unlabeled",
        fitting_data_source="disjoint_target_fit_subset_unlabeled",
        labels_used=False,
        target_data_used=True,
        target_labels_used=False,
        synthetic_perturbations_used=False,
        benchmark_selection_used=False,
        fit_eval_disjoint=True,
        leakage_risk="controlled",
        status="passed",
        notes=(
            "PseudoCal fit uses only unlabeled target-fit subsets with fixed threshold 0.90. "
            "Final evaluation is reported only on disjoint target-eval subsets."
        ),
    )
)

print("PseudoCal audit row added.")
print("Current number of audit rows:", len(audit_rows))

## MM-NLL-TS rerun

In [ ]:
This section reruns MM-NLL-TS under the frozen source-side robust calibration protocol.

MM-NLL-TS belongs to Regime C and uses:

- the official checkpoint,
- the clean labeled source calibration split,
- multiple fixed source-side perturbation views,
- one scalar temperature selected by minimizing the worst-case perturbed negative log-likelihood across those views.

The method does not use target data during fitting.

After fitting, the calibrated model is evaluated on:

- clean validation,
- all filtered ImageNet-C corruptions and severities.

This section keeps the perturbation set, temperature grid, and minimax objective explicit and frozen.

MMNLL_METHOD = "mmnll_ts"
MMNLL_REGIME = "Regime_C_source_robust"
MMNLL_OUTPUT_DIR = Path(PATHS["results_root"]) / "mmnll"
MMNLL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MMNLL_CONFIG = {
    "temperature_grid": [round(x, 3) for x in np.linspace(0.5, 2.5, 81)],
    "perturbation_views": [
        {
            "name": "blur_mild",
            "resize": 256,
            "center_crop": IMAGE_SIZE,
            "gaussian_blur_kernel_size": 3,
            "gaussian_blur_sigma": 0.6,
            "jitter_brightness": 0.00,
            "jitter_contrast": 0.00,
            "jitter_saturation": 0.00,
            "jitter_hue": 0.00,
        },
        {
            "name": "blur_medium",
            "resize": 256,
            "center_crop": IMAGE_SIZE,
            "gaussian_blur_kernel_size": 5,
            "gaussian_blur_sigma": 1.0,
            "jitter_brightness": 0.00,
            "jitter_contrast": 0.00,
            "jitter_saturation": 0.00,
            "jitter_hue": 0.00,
        },
        {
            "name": "brightness_contrast",
            "resize": 256,
            "center_crop": IMAGE_SIZE,
            "gaussian_blur_kernel_size": None,
            "gaussian_blur_sigma": None,
            "jitter_brightness": 0.20,
            "jitter_contrast": 0.20,
            "jitter_saturation": 0.05,
            "jitter_hue": 0.02,
        },
        {
            "name": "blur_plus_jitter",
            "resize": 256,
            "center_crop": IMAGE_SIZE,
            "gaussian_blur_kernel_size": 5,
            "gaussian_blur_sigma": 1.0,
            "jitter_brightness": 0.15,
            "jitter_contrast": 0.15,
            "jitter_saturation": 0.05,
            "jitter_hue": 0.02,
        },
    ],
    "objective": "minimize_worst_case_perturbed_nll",
}

print("MM-NLL output dir:", MMNLL_OUTPUT_DIR)
print("Number of temperature candidates:", len(MMNLL_CONFIG["temperature_grid"]))
print("Perturbation views:")
for view in MMNLL_CONFIG["perturbation_views"]:
    print(" -", view["name"])

def build_mmnll_transform(view_cfg: dict):
    t_list = [
        transforms.Resize(view_cfg["resize"]),
        transforms.CenterCrop(view_cfg["center_crop"]),
    ]

    if view_cfg["gaussian_blur_kernel_size"] is not None:
        t_list.append(
            transforms.GaussianBlur(
                kernel_size=view_cfg["gaussian_blur_kernel_size"],
                sigma=view_cfg["gaussian_blur_sigma"]
            )
        )

    if any([
        view_cfg["jitter_brightness"] > 0,
        view_cfg["jitter_contrast"] > 0,
        view_cfg["jitter_saturation"] > 0,
        view_cfg["jitter_hue"] > 0,
    ]):
        t_list.append(
            transforms.ColorJitter(
                brightness=view_cfg["jitter_brightness"],
                contrast=view_cfg["jitter_contrast"],
                saturation=view_cfg["jitter_saturation"],
                hue=view_cfg["jitter_hue"],
            )
        )

    t_list.extend([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    return transforms.Compose(t_list)

print("MM-NLL transform factory defined.")

mmnll_calib_loaders = {}

for view_cfg in MMNLL_CONFIG["perturbation_views"]:
    view_name = view_cfg["name"]
    view_transform = build_mmnll_transform(view_cfg)

    full_dataset_view = datasets.ImageFolder(
        root=str(CLEAN_TRAIN_DIR),
        transform=view_transform
    )

    assert full_dataset_view.classes == imagenet100_classes, (
        f"Class order mismatch for MM-NLL perturbation view: {view_name}"
    )

    calib_subset_view = Subset(full_dataset_view, calib_idx)

    calib_loader_view = DataLoader(
        calib_subset_view,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    mmnll_calib_loaders[view_name] = calib_loader_view

print("Built MM-NLL perturbed calibration loaders:")
print(list(mmnll_calib_loaders.keys()))

def fit_mmnll_temperature_grid(model: nn.Module, calib_loaders_by_view: dict, temperature_grid: list, device=DEVICE):
    """
    For each perturbation view, extract calibration logits once.
    Then select T that minimizes the worst-case NLL across views.
    """
    logits_by_view = {}
    labels_reference = None

    # Extract logits for each perturbation view
    for view_name, loader in calib_loaders_by_view.items():
        logits_view, labels_view = get_logits_labels(model, loader, device=device)

        if labels_reference is None:
            labels_reference = labels_view
        else:
            assert torch.equal(labels_reference, labels_view), (
                f"Label mismatch across perturbation views for MM-NLL: {view_name}"
            )

        logits_by_view[view_name] = logits_view

    labels_reference = labels_reference.clone()

    grid_rows = []
    best_T = None
    best_worst_nll = None

    for T in temperature_grid:
        view_nlls = {}
        for view_name, logits_view in logits_by_view.items():
            scaled_logits = apply_temperature(logits_view, T)
            view_nll = nll_from_logits(scaled_logits, labels_reference)
            view_nlls[view_name] = view_nll

        worst_case_nll = max(view_nlls.values())
        mean_view_nll = float(np.mean(list(view_nlls.values())))

        row = {
            "temperature": float(T),
            "worst_case_nll": float(worst_case_nll),
            "mean_view_nll": float(mean_view_nll),
        }
        for view_name, v in view_nlls.items():
            row[f"nll_{view_name}"] = float(v)

        grid_rows.append(row)

        if (best_worst_nll is None) or (worst_case_nll < best_worst_nll):
            best_worst_nll = worst_case_nll
            best_T = float(T)

    grid_df = pd.DataFrame(grid_rows)

    return {
        "best_temperature": best_T,
        "best_worst_case_nll": float(best_worst_nll),
        "grid_df": grid_df,
        "num_fit_samples": int(len(labels_reference)),
        "perturbation_views": list(logits_by_view.keys()),
    }

print("MM-NLL fitting helper defined.")

mmnll_protocol_summary = {
    "method": MMNLL_METHOD,
    "regime": MMNLL_REGIME,
    "fit_source": "source_calibration_with_fixed_perturbation_views",
    "config": {
        "objective": MMNLL_CONFIG["objective"],
        "num_temperature_candidates": len(MMNLL_CONFIG["temperature_grid"]),
        "temperature_grid_min": min(MMNLL_CONFIG["temperature_grid"]),
        "temperature_grid_max": max(MMNLL_CONFIG["temperature_grid"]),
        "perturbation_view_names": [v["name"] for v in MMNLL_CONFIG["perturbation_views"]],
    },
    "note": (
        "MM-NLL-TS selects the scalar temperature that minimizes the worst-case "
        "perturbed calibration NLL across fixed source-side perturbation views."
    ),
}

mmnll_protocol_path = MMNLL_OUTPUT_DIR / "mmnll_protocol_summary.json"
save_json(mmnll_protocol_summary, mmnll_protocol_path)

print("Saved MM-NLL protocol summary to:")
print(mmnll_protocol_path)

mmnll_all_rows = []
mmnll_fit_rows = []

for seed in PROTOCOL["seeds"]:
    print("=" * 80)
    print(f"Running MM-NLL-TS for seed {seed}")
    print("=" * 80)

    set_seed(seed)
    model = load_resnet18_checkpoint(seed, device=DEVICE)
    checkpoint_name = PROTOCOL["checkpoint_rule"]


    # Fit on fixed perturbed source calibration views

    fit_result = fit_mmnll_temperature_grid(
        model=model,
        calib_loaders_by_view=mmnll_calib_loaders,
        temperature_grid=MMNLL_CONFIG["temperature_grid"],
        device=DEVICE,
    )

    fitted_T = fit_result["best_temperature"]
    grid_df = fit_result["grid_df"]

    grid_csv_path = MMNLL_OUTPUT_DIR / f"mmnll_seed_{seed}_grid.csv"
    save_dataframe(grid_df, grid_csv_path)

    mmnll_fit_rows.append({
        "method": MMNLL_METHOD,
        "regime": MMNLL_REGIME,
        "seed": seed,
        "temperature": fitted_T,
        "best_worst_case_nll": fit_result["best_worst_case_nll"],
        "num_fit_samples": fit_result["num_fit_samples"],
        "num_views": len(fit_result["perturbation_views"]),
        "fit_source": "source_calibration_with_fixed_perturbation_views",
        "checkpoint_name": checkpoint_name,
        "grid_csv": str(grid_csv_path),
    })

    print(f"[Seed {seed}] Fitted MM-NLL temperature: {fitted_T:.6f}")

    seed_rows = []


    # Clean validation evaluation

    clean_logits, clean_labels = get_logits_labels(model, clean_val_loader, device=DEVICE)
    clean_logits_mmnll = apply_temperature(clean_logits, fitted_T)
    clean_metrics_mmnll = metrics_from_logits(clean_logits_mmnll, clean_labels)

    seed_rows.extend(
        metrics_dict_to_long_rows(
            metrics_dict=clean_metrics_mmnll,
            method=MMNLL_METHOD,
            regime=MMNLL_REGIME,
            seed=seed,
            split="clean_val",
            corruption=None,
            severity=None,
            checkpoint_name=checkpoint_name,
            fit_source="source_calibration_with_fixed_perturbation_views",
            eval_source="clean_val",
        )
    )

    print(f"[Seed {seed}] Clean validation done.")


    # ImageNet-C evaluation

    for corruption in PROTOCOL["corruptions"]:
        for severity in PROTOCOL["severities"]:
            corr_loader = build_imagenetc_loader(corruption=corruption, severity=severity)
            corr_logits, corr_labels = get_logits_labels(model, corr_loader, device=DEVICE)
            corr_logits_mmnll = apply_temperature(corr_logits, fitted_T)
            corr_metrics_mmnll = metrics_from_logits(corr_logits_mmnll, corr_labels)

            seed_rows.extend(
                metrics_dict_to_long_rows(
                    metrics_dict=corr_metrics_mmnll,
                    method=MMNLL_METHOD,
                    regime=MMNLL_REGIME,
                    seed=seed,
                    split="imagenet_c",
                    corruption=corruption,
                    severity=severity,
                    checkpoint_name=checkpoint_name,
                    fit_source="source_calibration_with_fixed_perturbation_views",
                    eval_source="imagenet_c_filtered",
                )
            )

        print(f"[Seed {seed}] Finished corruption: {corruption}")

    seed_df = pd.DataFrame(seed_rows)
    seed_csv_path = MMNLL_OUTPUT_DIR / f"mmnll_seed_{seed}_long.csv"
    save_dataframe(seed_df, seed_csv_path)

    print(f"[Seed {seed}] Saved per-seed MM-NLL results to:")
    print(seed_csv_path)

    mmnll_all_rows.extend(seed_rows)

    del model
    torch.cuda.empty_cache()

mmnll_fit_df = pd.DataFrame(mmnll_fit_rows)
mmnll_fit_csv_path = MMNLL_OUTPUT_DIR / "mmnll_fit_summary.csv"
save_dataframe(mmnll_fit_df, mmnll_fit_csv_path)

print("Saved MM-NLL fit summary to:")
print(mmnll_fit_csv_path)

display(mmnll_fit_df)

mmnll_all_df = pd.DataFrame(mmnll_all_rows)

print("Combined MM-NLL result shape:", mmnll_all_df.shape)
print("Columns:", list(mmnll_all_df.columns))

mmnll_all_csv_path = MMNLL_OUTPUT_DIR / "mmnll_all_seeds_long.csv"
save_dataframe(mmnll_all_df, mmnll_all_csv_path)

print("Saved combined all-seeds MM-NLL results to:")
print(mmnll_all_csv_path)

group_cols = ["method", "regime", "split", "corruption", "severity", "metric"]

mmnll_agg_rows = []

for group_key, group_df in mmnll_all_df.groupby(group_cols, dropna=False):
    values = group_df["value"].values
    agg_stats = bootstrap_mean_ci(values, n_boot=1000, ci=95.0, seed=123)

    row = dict(zip(group_cols, group_key))
    row.update({
        "n_seeds": len(group_df),
        "mean": agg_stats["mean"],
        "ci_lower": agg_stats["ci_lower"],
        "ci_upper": agg_stats["ci_upper"],
    })
    mmnll_agg_rows.append(row)

mmnll_agg_df = pd.DataFrame(mmnll_agg_rows)

print("Aggregated MM-NLL result shape:", mmnll_agg_df.shape)
mmnll_agg_df.head()

mmnll_agg_csv_path = MMNLL_OUTPUT_DIR / "mmnll_aggregated.csv"
save_dataframe(mmnll_agg_df, mmnll_agg_csv_path)

print("Saved aggregated MM-NLL results to:")
print(mmnll_agg_csv_path)

mmnll_summary = {
    "method": MMNLL_METHOD,
    "regime": MMNLL_REGIME,
    "fit_required": True,
    "fit_source": "source_calibration_with_fixed_perturbation_views",
    "objective": MMNLL_CONFIG["objective"],
    "seeds": PROTOCOL["seeds"],
    "checkpoint_rule": PROTOCOL["checkpoint_rule"],
    "corruptions": PROTOCOL["corruptions"],
    "severities": PROTOCOL["severities"],
    "protocol_summary_json": str(mmnll_protocol_path),
    "fit_summary_csv": str(mmnll_fit_csv_path),
    "per_seed_grid_csvs": [
        str(MMNLL_OUTPUT_DIR / f"mmnll_seed_{seed}_grid.csv")
        for seed in PROTOCOL["seeds"]
    ],
    "per_seed_csvs": [
        str(MMNLL_OUTPUT_DIR / f"mmnll_seed_{seed}_long.csv")
        for seed in PROTOCOL["seeds"]
    ],
    "combined_csv": str(mmnll_all_csv_path),
    "aggregated_csv": str(mmnll_agg_csv_path),
}

mmnll_summary_path = MMNLL_OUTPUT_DIR / "mmnll_summary.json"
save_json(mmnll_summary, mmnll_summary_path)

print("Saved MM-NLL summary metadata to:")
print(mmnll_summary_path)

audit_rows.append(
    make_audit_row(
        method="mmnll_ts",
        regime="Regime_C_source_robust",
        fitting_data_source="source_calibration_with_fixed_perturbation_views",
        labels_used=True,
        target_data_used=False,
        target_labels_used=False,
        synthetic_perturbations_used=True,
        benchmark_selection_used=False,
        fit_eval_disjoint=True,
        leakage_risk="low",
        status="passed",
        notes=(
            "MM-NLL-TS uses only labeled source calibration data under fixed source-side perturbation views "
            "and selects T by minimizing worst-case perturbed NLL over a frozen temperature grid."
        ),
    )
)

print("MM-NLL audit row added.")
print("Current number of audit rows:", len(audit_rows))

## Regime-specific comparison tables

In [ ]:
This section creates the official within-regime comparison tables.

The comparisons are kept separate by regime:

- **Regime A — Clean-source post-hoc calibration**  
  baseline, TS, TS-P

- **Regime B — Target-unlabeled adaptation**  
  baseline, PseudoCal

- **Regime C — Source-side robust calibration**  
  baseline, MM-NLL-TS

The purpose of this section is to provide fair within-regime summaries without mixing methods that use different fitting information.

No global cross-regime leaderboard is produced here.


# Helper functions for regime comparison tables

COMPARISON_METRICS = ["accuracy", "nll", "brier", "ece15", "aece15"]

def prepare_clean_table(df: pd.DataFrame, method_order: list, split_name: str):
    table_df = df[
        (df["split"] == split_name) &
        (df["metric"].isin(COMPARISON_METRICS)) &
        (df["method"].isin(method_order))
    ].copy()

    table_df["method"] = pd.Categorical(table_df["method"], categories=method_order, ordered=True)
    table_df = table_df.sort_values(["method", "metric"])

    pivot_df = table_df.pivot_table(
        index="method",
        columns="metric",
        values="mean",
        aggfunc="first"
    )

    pivot_df = pivot_df.reindex(method_order)
    return pivot_df.reset_index()

def prepare_imagenetc_overall_table(df: pd.DataFrame, method_order: list, split_name: str):
    table_df = df[
        (df["split"] == split_name) &
        (df["metric"].isin(COMPARISON_METRICS)) &
        (df["method"].isin(method_order))
    ].copy()

    # average over corruption x severity rows for each method x metric
    summary_df = (
        table_df
        .groupby(["method", "metric"], as_index=False)["mean"]
        .mean()
    )

    summary_df["method"] = pd.Categorical(summary_df["method"], categories=method_order, ordered=True)
    summary_df = summary_df.sort_values(["method", "metric"])

    pivot_df = summary_df.pivot_table(
        index="method",
        columns="metric",
        values="mean",
        aggfunc="first"
    )

    pivot_df = pivot_df.reindex(method_order)
    return pivot_df.reset_index()

def save_table_csv_and_display(df: pd.DataFrame, path: Path, title: str):
    save_dataframe(df, path)
    print(title)
    print("Saved to:", path)
    display(df)

regime_a_methods = ["baseline", "ts", "tsp"]

regime_a_agg_df = pd.concat([
    baseline_agg_df[baseline_agg_df["method"] == "baseline"],
    ts_agg_df[ts_agg_df["method"] == "ts"],
    tsp_agg_df[tsp_agg_df["method"] == "tsp"],
], ignore_index=True)

print("Regime A aggregated pool shape:", regime_a_agg_df.shape)
print("Methods in Regime A pool:", sorted(regime_a_agg_df["method"].dropna().unique()))

regime_a_clean_table = prepare_clean_table(
    df=regime_a_agg_df,
    method_order=regime_a_methods,
    split_name="clean_val"
)

regime_a_imagenetc_table = prepare_imagenetc_overall_table(
    df=regime_a_agg_df,
    method_order=regime_a_methods,
    split_name="imagenet_c"
)

regime_a_clean_table_path = Path(PATHS["results_root"]) / "tables" / "regime_a_clean_val_table.csv"
regime_a_imagenetc_table_path = Path(PATHS["results_root"]) / "tables" / "regime_a_imagenetc_overall_table.csv"

save_table_csv_and_display(
    regime_a_clean_table,
    regime_a_clean_table_path,
    "Regime A — Clean validation comparison"
)

save_table_csv_and_display(
    regime_a_imagenetc_table,
    regime_a_imagenetc_table_path,
    "Regime A — ImageNet-C overall comparison"
)

regime_c_methods = ["baseline", "mmnll_ts"]

baseline_regime_c_subset = baseline_agg_df[baseline_agg_df["method"] == "baseline"].copy()
baseline_regime_c_subset["regime"] = "Regime_C_source_robust"

regime_c_agg_df = pd.concat([
    baseline_regime_c_subset,
    mmnll_agg_df[mmnll_agg_df["method"] == "mmnll_ts"],
], ignore_index=True)

print("Regime C aggregated pool shape:", regime_c_agg_df.shape)
print("Methods in Regime C pool:", sorted(regime_c_agg_df["method"].dropna().unique()))

regime_c_clean_table = prepare_clean_table(
    df=regime_c_agg_df,
    method_order=regime_c_methods,
    split_name="clean_val"
)

regime_c_imagenetc_table = prepare_imagenetc_overall_table(
    df=regime_c_agg_df,
    method_order=regime_c_methods,
    split_name="imagenet_c"
)

regime_c_clean_table_path = Path(PATHS["results_root"]) / "tables" / "regime_c_clean_val_table.csv"
regime_c_imagenetc_table_path = Path(PATHS["results_root"]) / "tables" / "regime_c_imagenetc_overall_table.csv"

save_table_csv_and_display(
    regime_c_clean_table,
    regime_c_clean_table_path,
    "Regime C — Clean validation comparison"
)

save_table_csv_and_display(
    regime_c_imagenetc_table,
    regime_c_imagenetc_table_path,
    "Regime C — ImageNet-C overall comparison"
)

regime_table_summary = {
    "comparison_metrics": COMPARISON_METRICS,
    "regime_a": {
        "methods": regime_a_methods,
        "clean_val_table": str(regime_a_clean_table_path),
        "imagenetc_overall_table": str(regime_a_imagenetc_table_path),
    },

    "regime_c": {
        "methods": regime_c_methods,
        "clean_val_table": str(regime_c_clean_table_path),
        "imagenetc_overall_table": str(regime_c_imagenetc_table_path),
    },
    "note": (
        "These are within-regime comparison tables only. "
        "No global cross-regime leaderboard is produced."
    ),
}

regime_table_summary_path = Path(PATHS["results_root"]) / "tables" / "regime_table_summary.json"
save_json(regime_table_summary, regime_table_summary_path)

print("Saved regime table summary to:")
print(regime_table_summary_path)

## Fairness repair for Regime B baseline

In [ ]:
This section aligns the Regime B baseline comparison with the PseudoCal evaluation protocol.

PseudoCal is evaluated only on disjoint target-eval subsets.  
Therefore, to make the Regime B comparison fair, the baseline must also be evaluated on the exact same target-eval subsets.

This section reruns the uncalibrated baseline on the same disjoint target-eval subsets used by PseudoCal and saves the aligned results for Regime B comparison.

BASELINE_B_TARGETEVAL_METHOD = "baseline"
BASELINE_B_TARGETEVAL_REGIME = "Regime_B_target_unlabeled"
BASELINE_B_TARGETEVAL_OUTPUT_DIR = Path(PATHS["results_root"]) / "baseline"

print("Regime B aligned baseline evaluation will be saved under:")
print(BASELINE_B_TARGETEVAL_OUTPUT_DIR)


# Run baseline on disjoint target-eval subsets

baseline_b_targeteval_rows = []

for seed in PROTOCOL["seeds"]:
    print("=" * 80)
    print(f"Running Regime B aligned baseline for seed {seed}")
    print("=" * 80)

    set_seed(seed)
    model = load_resnet18_checkpoint(seed, device=DEVICE)
    checkpoint_name = PROTOCOL["checkpoint_rule"]

    seed_rows = []

    # Clean validation reference
    clean_logits, clean_labels = get_logits_labels(model, clean_val_loader, device=DEVICE)
    clean_metrics = metrics_from_logits(clean_logits, clean_labels)

    seed_rows.extend(
        metrics_dict_to_long_rows(
            metrics_dict=clean_metrics,
            method=BASELINE_B_TARGETEVAL_METHOD,
            regime=BASELINE_B_TARGETEVAL_REGIME,
            seed=seed,
            split="clean_val",
            corruption=None,
            severity=None,
            checkpoint_name=checkpoint_name,
            fit_source="none",
            eval_source="clean_val",
        )
    )

    # Evaluate on the exact same target-eval subsets used by PseudoCal
    for corruption in PROTOCOL["corruptions"]:
        for severity in PROTOCOL["severities"]:
            _fit_subset, eval_subset, fit_idx, eval_idx, total_n = build_pseudocal_fit_eval_subsets(
                corruption=corruption,
                severity=severity,
            )

            eval_loader = DataLoader(
                eval_subset,
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=NUM_WORKERS,
                pin_memory=PIN_MEMORY,
            )

            eval_logits, eval_labels = get_logits_labels(model, eval_loader, device=DEVICE)
            eval_metrics = metrics_from_logits(eval_logits, eval_labels)

            seed_rows.extend(
                metrics_dict_to_long_rows(
                    metrics_dict=eval_metrics,
                    method=BASELINE_B_TARGETEVAL_METHOD,
                    regime=BASELINE_B_TARGETEVAL_REGIME,
                    seed=seed,
                    split="imagenet_c_target_eval",
                    corruption=corruption,
                    severity=severity,
                    checkpoint_name=checkpoint_name,
                    fit_source="none",
                    eval_source="disjoint_target_eval_subset",
                )
            )

        print(f"[Seed {seed}] Finished corruption: {corruption}")

    seed_df = pd.DataFrame(seed_rows)
    seed_csv_path = BASELINE_B_TARGETEVAL_OUTPUT_DIR / f"baseline_regime_b_target_eval_seed_{seed}_long.csv"
    save_dataframe(seed_df, seed_csv_path)

    print(f"[Seed {seed}] Saved aligned Regime B baseline results to:")
    print(seed_csv_path)

    baseline_b_targeteval_rows.extend(seed_rows)

    del model
    torch.cuda.empty_cache()


#  Combine and aggregate aligned Regime B baseline

baseline_b_targeteval_df = pd.DataFrame(baseline_b_targeteval_rows)

baseline_b_targeteval_all_csv_path = (
    BASELINE_B_TARGETEVAL_OUTPUT_DIR / "baseline_regime_b_target_eval_all_seeds_long.csv"
)
save_dataframe(baseline_b_targeteval_df, baseline_b_targeteval_all_csv_path)

print("Saved combined Regime B aligned baseline results to:")
print(baseline_b_targeteval_all_csv_path)

group_cols = ["method", "regime", "split", "corruption", "severity", "metric"]

baseline_b_targeteval_agg_rows = []

for group_key, group_df in baseline_b_targeteval_df.groupby(group_cols, dropna=False):
    values = group_df["value"].values
    agg_stats = bootstrap_mean_ci(values, n_boot=1000, ci=95.0, seed=123)

    row = dict(zip(group_cols, group_key))
    row.update({
        "n_seeds": len(group_df),
        "mean": agg_stats["mean"],
        "ci_lower": agg_stats["ci_lower"],
        "ci_upper": agg_stats["ci_upper"],
    })
    baseline_b_targeteval_agg_rows.append(row)

baseline_b_targeteval_agg_df = pd.DataFrame(baseline_b_targeteval_agg_rows)

baseline_b_targeteval_agg_csv_path = (
    BASELINE_B_TARGETEVAL_OUTPUT_DIR / "baseline_regime_b_target_eval_aggregated.csv"
)
save_dataframe(baseline_b_targeteval_agg_df, baseline_b_targeteval_agg_csv_path)

print("Saved aggregated Regime B aligned baseline results to:")
print(baseline_b_targeteval_agg_csv_path)


# Rebuild Regime B tables with aligned baseline

regime_b_methods = ["baseline", "pseudocal"]

regime_b_aligned_agg_df = pd.concat([
    baseline_b_targeteval_agg_df[baseline_b_targeteval_agg_df["method"] == "baseline"],
    pseudocal_agg_df[pseudocal_agg_df["method"] == "pseudocal"],
], ignore_index=True)

print("Aligned Regime B aggregated pool shape:", regime_b_aligned_agg_df.shape)
print("Methods in aligned Regime B pool:", sorted(regime_b_aligned_agg_df["method"].dropna().unique()))

regime_b_clean_table = prepare_clean_table(
    df=regime_b_aligned_agg_df,
    method_order=regime_b_methods,
    split_name="clean_val"
)

regime_b_targeteval_table = prepare_imagenetc_overall_table(
    df=regime_b_aligned_agg_df,
    method_order=regime_b_methods,
    split_name="imagenet_c_target_eval"
)

regime_b_clean_table_path = Path(PATHS["results_root"]) / "tables" / "regime_b_clean_val_table.csv"
regime_b_targeteval_table_path = Path(PATHS["results_root"]) / "tables" / "regime_b_target_eval_table.csv"

save_table_csv_and_display(
    regime_b_clean_table,
    regime_b_clean_table_path,
    "Regime B — Clean validation comparison (aligned baseline)"
)

save_table_csv_and_display(
    regime_b_targeteval_table,
    regime_b_targeteval_table_path,
    "Regime B — Target-eval comparison (aligned baseline)"
)

regime_b_alignment_summary = {
    "reason": (
        "PseudoCal is evaluated on disjoint target-eval subsets, so baseline was rerun on the same "
        "target-eval subsets to make the Regime B comparison fair."
    ),
    "baseline_combined_csv": str(baseline_b_targeteval_all_csv_path),
    "baseline_aggregated_csv": str(baseline_b_targeteval_agg_csv_path),
    "regime_b_clean_table": str(regime_b_clean_table_path),
    "regime_b_target_eval_table": str(regime_b_targeteval_table_path),
}

regime_b_alignment_summary_path = Path(PATHS["results_root"]) / "tables" / "regime_b_alignment_summary.json"
save_json(regime_b_alignment_summary, regime_b_alignment_summary_path)

print("Saved Regime B alignment summary to:")
print(regime_b_alignment_summary_path)

audit_rows.append(
    make_audit_row(
        method="baseline_regime_b_aligned_eval",
        regime="Regime_B_target_unlabeled",
        fitting_data_source="none",
        labels_used=False,
        target_data_used=False,
        target_labels_used=False,
        synthetic_perturbations_used=False,
        benchmark_selection_used=False,
        fit_eval_disjoint="matched_to_pseudocal_target_eval",
        leakage_risk="none",
        status="passed",
        notes=(
            "Additional baseline evaluation on the exact disjoint target-eval subsets used for PseudoCal, "
            "to ensure fair within-regime comparison in Regime B."
        ),
    )
)

print("Added Regime B fairness repair audit row.")
print("Current number of audit rows:", len(audit_rows))

##  Regime-specific plots

In [ ]:
This section generates the official regime-specific comparison plots.

The plots are kept separate by regime to preserve fairness and interpretability.

For each regime, the section produces:

- ECE15 vs severity
- NLL vs severity

The regimes are:

- **Regime A — Clean-source post-hoc calibration**
- **Regime B — Target-unlabeled adaptation**
- **Regime C — Source-side robust calibration**

No mixed cross-regime plot is produced here.

PLOTS_OUTPUT_DIR = Path(PATHS["results_root"]) / "figures"
PLOTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def make_severity_curve_df(df: pd.DataFrame, split_name: str, metric_name: str, method_order: list):
    sub = df[
        (df["split"] == split_name) &
        (df["metric"] == metric_name) &
        (df["method"].isin(method_order)) &
        (df["severity"].notna())
    ].copy()

    sub["severity"] = sub["severity"].astype(int)

    curve_df = (
        sub.groupby(["method", "severity"], as_index=False)["mean"]
        .mean()
    )

    curve_df["method"] = pd.Categorical(curve_df["method"], categories=method_order, ordered=True)
    curve_df = curve_df.sort_values(["method", "severity"])
    return curve_df

def plot_metric_vs_severity(curve_df: pd.DataFrame, methods: list, metric_name: str, title: str, save_path: Path):
    plt.figure(figsize=(8, 5))

    for method in methods:
        method_df = curve_df[curve_df["method"] == method]
        plt.plot(
            method_df["severity"],
            method_df["mean"],
            marker="o",
            label=method,
        )

    plt.xlabel("Severity")
    plt.ylabel(metric_name.upper())
    plt.title(title)
    plt.xticks([1, 2, 3, 4, 5])
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved figure to:")
    print(save_path)

print("Shared plotting helpers defined.")

regime_a_methods = ["baseline", "ts", "tsp"]

regime_a_ece_curve = make_severity_curve_df(
    df=regime_a_agg_df,
    split_name="imagenet_c",
    metric_name="ece15",
    method_order=regime_a_methods,
)

regime_a_nll_curve = make_severity_curve_df(
    df=regime_a_agg_df,
    split_name="imagenet_c",
    metric_name="nll",
    method_order=regime_a_methods,
)

regime_a_ece_path = PLOTS_OUTPUT_DIR / "regime_a_ece15_vs_severity.png"
regime_a_nll_path = PLOTS_OUTPUT_DIR / "regime_a_nll_vs_severity.png"

plot_metric_vs_severity(
    curve_df=regime_a_ece_curve,
    methods=regime_a_methods,
    metric_name="ece15",
    title="Regime A — ECE15 vs Severity",
    save_path=regime_a_ece_path,
)

plot_metric_vs_severity(
    curve_df=regime_a_nll_curve,
    methods=regime_a_methods,
    metric_name="nll",
    title="Regime A — NLL vs Severity",
    save_path=regime_a_nll_path,
)

regime_b_methods = ["baseline", "pseudocal"]

regime_b_ece_curve = make_severity_curve_df(
    df=regime_b_aligned_agg_df,
    split_name="imagenet_c_target_eval",
    metric_name="ece15",
    method_order=regime_b_methods,
)

regime_b_nll_curve = make_severity_curve_df(
    df=regime_b_aligned_agg_df,
    split_name="imagenet_c_target_eval",
    metric_name="nll",
    method_order=regime_b_methods,
)

regime_b_ece_path = PLOTS_OUTPUT_DIR / "regime_b_ece15_vs_severity.png"
regime_b_nll_path = PLOTS_OUTPUT_DIR / "regime_b_nll_vs_severity.png"

plot_metric_vs_severity(
    curve_df=regime_b_ece_curve,
    methods=regime_b_methods,
    metric_name="ece15",
    title="Regime B — ECE15 vs Severity (Target-Eval)",
    save_path=regime_b_ece_path,
)

plot_metric_vs_severity(
    curve_df=regime_b_nll_curve,
    methods=regime_b_methods,
    metric_name="nll",
    title="Regime B — NLL vs Severity (Target-Eval)",
    save_path=regime_b_nll_path,
)

regime_c_methods = ["baseline", "mmnll_ts"]

regime_c_ece_curve = make_severity_curve_df(
    df=regime_c_agg_df,
    split_name="imagenet_c",
    metric_name="ece15",
    method_order=regime_c_methods,
)

regime_c_nll_curve = make_severity_curve_df(
    df=regime_c_agg_df,
    split_name="imagenet_c",
    metric_name="nll",
    method_order=regime_c_methods,
)

regime_c_ece_path = PLOTS_OUTPUT_DIR / "regime_c_ece15_vs_severity.png"
regime_c_nll_path = PLOTS_OUTPUT_DIR / "regime_c_nll_vs_severity.png"

plot_metric_vs_severity(
    curve_df=regime_c_ece_curve,
    methods=regime_c_methods,
    metric_name="ece15",
    title="Regime C — ECE15 vs Severity",
    save_path=regime_c_ece_path,
)

plot_metric_vs_severity(
    curve_df=regime_c_nll_curve,
    methods=regime_c_methods,
    metric_name="nll",
    title="Regime C — NLL vs Severity",
    save_path=regime_c_nll_path,
)

section14_plot_summary = {
    "regime_a": {
        "methods": ["baseline", "ts", "tsp"],
        "ece15_plot": str(regime_a_ece_path),
        "nll_plot": str(regime_a_nll_path),
    },
    "regime_b": {
        "methods": ["baseline", "pseudocal"],
        "ece15_plot": str(regime_b_ece_path),
        "nll_plot": str(regime_b_nll_path),
        "note": "Uses aligned target-eval baseline for fair Regime B comparison.",
    },
    "regime_c": {
        "methods": ["baseline", "mmnll_ts"],
        "ece15_plot": str(regime_c_ece_path),
        "nll_plot": str(regime_c_nll_path),
    },
}

section14_plot_summary_path = PLOTS_OUTPUT_DIR / "section14_plot_summary.json"
save_json(section14_plot_summary, section14_plot_summary_path)

print("Saved Section 14 plot summary to:")
print(section14_plot_summary_path)

## Reliability diagrams

In [ ]:
This section generates reliability diagrams for the official methods under representative evaluation cases.

The purpose of this section is to provide a qualitative view of calibration behavior beyond scalar summary metrics.

The methods included are:

- baseline
- TS
- TS-P
- PseudoCal
- MM-NLL-TS

The representative cases are:

- clean validation
- one severe noise condition
- one severe blur condition

These diagrams are intended for interpretation within the regime-aware protocol and not as a mixed global leaderboard.

def compute_reliability_bins(logits: torch.Tensor, labels: torch.Tensor, n_bins: int = 15):
    probs = torch.softmax(logits, dim=1)
    confidences, predictions = torch.max(probs, dim=1)
    accuracies = predictions.eq(labels).float()

    bin_edges = torch.linspace(0, 1, n_bins + 1)
    bin_centers = []
    bin_acc = []
    bin_conf = []
    bin_counts = []

    for i in range(n_bins):
        lower = bin_edges[i]
        upper = bin_edges[i + 1]

        in_bin = (confidences > lower) & (confidences <= upper)
        count = int(in_bin.sum().item())

        if count > 0:
            acc = accuracies[in_bin].mean().item()
            conf = confidences[in_bin].mean().item()
        else:
            acc = np.nan
            conf = np.nan

        bin_centers.append(float((lower + upper) / 2))
        bin_acc.append(acc)
        bin_conf.append(conf)
        bin_counts.append(count)

    return pd.DataFrame({
        "bin_center": bin_centers,
        "bin_accuracy": bin_acc,
        "bin_confidence": bin_conf,
        "bin_count": bin_counts,
    })

def plot_reliability_diagram_from_logits(
    logits: torch.Tensor,
    labels: torch.Tensor,
    title: str,
    save_path: Path,
    n_bins: int = 15,
):
    rel_df = compute_reliability_bins(logits, labels, n_bins=n_bins)

    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1, label="Perfect calibration")
    plt.plot(rel_df["bin_confidence"], rel_df["bin_accuracy"], marker="o", label="Model")

    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved figure to:")
    print(save_path)

    return rel_df

print("Reliability diagram helpers defined.")

RELIABILITY_OUTPUT_DIR = Path(PATHS["results_root"]) / "figures" / "reliability"
RELIABILITY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RELIABILITY_CASES = {
    "clean": {
        "type": "clean_val",
        "corruption": None,
        "severity": None,
    },
    "severe_noise": {
        "type": "imagenet_c",
        "corruption": "gaussian_noise",
        "severity": 5,
    },
    "severe_blur": {
        "type": "imagenet_c",
        "corruption": "glass_blur",
        "severity": 5,
    },
}

RELIABILITY_METHODS = ["baseline", "ts", "tsp", "pseudocal", "mmnll_ts"]

print("Reliability methods:", RELIABILITY_METHODS)
print("Reliability cases:", RELIABILITY_CASES)

def get_fitted_temperature_for_method_seed(method: str, seed: int, corruption: str = None, severity: int = None) -> float:
    if method == "baseline":
        return 1.0

    elif method == "ts":
        row = ts_fit_df[ts_fit_df["seed"] == seed]
        assert len(row) == 1, f"Expected one TS fit row for seed {seed}"
        return float(row.iloc[0]["temperature"])

    elif method == "tsp":
        row = tsp_fit_df[tsp_fit_df["seed"] == seed]
        assert len(row) == 1, f"Expected one TS-P fit row for seed {seed}"
        return float(row.iloc[0]["temperature"])

    elif method == "mmnll_ts":
        row = mmnll_fit_df[mmnll_fit_df["seed"] == seed]
        assert len(row) == 1, f"Expected one MM-NLL fit row for seed {seed}"
        return float(row.iloc[0]["temperature"])

    elif method == "pseudocal":
        assert corruption is not None and severity is not None, (
            "PseudoCal temperature lookup requires corruption and severity."
        )
        row = pseudocal_fit_df[
            (pseudocal_fit_df["seed"] == seed) &
            (pseudocal_fit_df["corruption"] == corruption) &
            (pseudocal_fit_df["severity"] == severity)
        ]
        assert len(row) == 1, (
            f"Expected one PseudoCal fit row for seed={seed}, corruption={corruption}, severity={severity}"
        )
        return float(row.iloc[0]["temperature"])

    else:
        raise ValueError(f"Unknown method: {method}")

print("Method-specific temperature lookup helper defined.")

def get_reliability_case_loader(case_name: str):
    case_cfg = RELIABILITY_CASES[case_name]

    if case_cfg["type"] == "clean_val":
        return clean_val_loader

    elif case_cfg["type"] == "imagenet_c":
        corruption = case_cfg["corruption"]
        severity = case_cfg["severity"]
        return build_imagenetc_loader(corruption=corruption, severity=severity)

    else:
        raise ValueError(f"Unknown reliability case type: {case_cfg['type']}")

print("Reliability case loader helper defined.")

RELIABILITY_REFERENCE_SEED = PROTOCOL["seeds"][0]

set_seed(RELIABILITY_REFERENCE_SEED)
reliability_model = load_resnet18_checkpoint(RELIABILITY_REFERENCE_SEED, device=DEVICE)

reliability_records = []

for method in RELIABILITY_METHODS:
    for case_name, case_cfg in RELIABILITY_CASES.items():
        print("=" * 80)
        print(f"Reliability diagram — method={method}, case={case_name}")
        print("=" * 80)

        loader = get_reliability_case_loader(case_name)
        logits, labels = get_logits_labels(reliability_model, loader, device=DEVICE)

        if method == "pseudocal":
            if case_cfg["type"] == "imagenet_c":
                fitted_T = get_fitted_temperature_for_method_seed(
                    method=method,
                    seed=RELIABILITY_REFERENCE_SEED,
                    corruption=case_cfg["corruption"],
                    severity=case_cfg["severity"],
                )
            else:
                fitted_T = 1.0
        else:
            fitted_T = get_fitted_temperature_for_method_seed(
                method=method,
                seed=RELIABILITY_REFERENCE_SEED,
            )

        scaled_logits = apply_temperature(logits, fitted_T)

        figure_name = f"reliability_{method}_{case_name}.png"
        figure_path = RELIABILITY_OUTPUT_DIR / figure_name

        rel_df = plot_reliability_diagram_from_logits(
            logits=scaled_logits,
            labels=labels,
            title=f"Reliability Diagram — {method} — {case_name}",
            save_path=figure_path,
            n_bins=15,
        )

        rel_csv_path = RELIABILITY_OUTPUT_DIR / f"reliability_{method}_{case_name}.csv"
        save_dataframe(rel_df, rel_csv_path)

        record = {
            "method": method,
            "case": case_name,
            "seed": RELIABILITY_REFERENCE_SEED,
            "temperature": fitted_T,
            "figure_path": str(figure_path),
            "csv_path": str(rel_csv_path),
        }

        if case_cfg["type"] == "imagenet_c":
            record["corruption"] = case_cfg["corruption"]
            record["severity"] = case_cfg["severity"]

        reliability_records.append(record)

reliability_summary_df = pd.DataFrame(reliability_records)
reliability_summary_csv_path = RELIABILITY_OUTPUT_DIR / "reliability_summary.csv"
save_dataframe(reliability_summary_df, reliability_summary_csv_path)

reliability_summary_json = {
    "reference_seed": RELIABILITY_REFERENCE_SEED,
    "methods": RELIABILITY_METHODS,
    "cases": RELIABILITY_CASES,
    "summary_csv": str(reliability_summary_csv_path),
}

reliability_summary_json_path = RELIABILITY_OUTPUT_DIR / "reliability_summary.json"
save_json(reliability_summary_json, reliability_summary_json_path)

print("Saved reliability summary CSV to:")
print(reliability_summary_csv_path)

print("Saved reliability summary JSON to:")
print(reliability_summary_json_path)

display(reliability_summary_df)

## Confidence histograms

In [ ]:
This section generates confidence histograms for the official methods under representative evaluation cases.

The purpose of this section is to visualize how confidence is distributed for correct and incorrect predictions.

The methods included are:

- baseline
- TS
- TS-P
- PseudoCal
- MM-NLL-TS

The representative cases are:

- clean validation
- one severe noise condition
- one severe blur condition

These histograms complement the reliability diagrams by showing how confidence mass shifts across methods and evaluation settings.

CONF_HIST_OUTPUT_DIR = Path(PATHS["results_root"]) / "figures" / "confidence_histograms"
CONF_HIST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def compute_confidence_dataframe(logits: torch.Tensor, labels: torch.Tensor) -> pd.DataFrame:
    probs = torch.softmax(logits, dim=1)
    confidences, preds = torch.max(probs, dim=1)
    correct = preds.eq(labels)

    return pd.DataFrame({
        "confidence": confidences.detach().cpu().numpy(),
        "correct": correct.detach().cpu().numpy().astype(int),
    })

def plot_confidence_histogram_from_logits(
    logits: torch.Tensor,
    labels: torch.Tensor,
    title: str,
    save_path: Path,
    bins: int = 15,
):
    conf_df = compute_confidence_dataframe(logits, labels)

    conf_correct = conf_df[conf_df["correct"] == 1]["confidence"].values
    conf_incorrect = conf_df[conf_df["correct"] == 0]["confidence"].values

    plt.figure(figsize=(7, 5))
    plt.hist(conf_correct, bins=bins, alpha=0.6, density=True, label="Correct")
    plt.hist(conf_incorrect, bins=bins, alpha=0.6, density=True, label="Incorrect")

    plt.xlabel("Confidence")
    plt.ylabel("Density")
    plt.title(title)
    plt.xlim(0, 1)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved figure to:")
    print(save_path)

    return conf_df

print("Confidence histogram helpers defined.")

CONF_HIST_METHODS = ["baseline", "ts", "tsp", "pseudocal", "mmnll_ts"]
CONF_HIST_CASES = RELIABILITY_CASES
CONF_HIST_REFERENCE_SEED = PROTOCOL["seeds"][0]

print("Confidence histogram methods:", CONF_HIST_METHODS)
print("Confidence histogram cases:", CONF_HIST_CASES)
print("Reference seed:", CONF_HIST_REFERENCE_SEED)

set_seed(CONF_HIST_REFERENCE_SEED)
conf_hist_model = load_resnet18_checkpoint(CONF_HIST_REFERENCE_SEED, device=DEVICE)

confidence_hist_records = []

for method in CONF_HIST_METHODS:
    for case_name, case_cfg in CONF_HIST_CASES.items():
        print("=" * 80)
        print(f"Confidence histogram — method={method}, case={case_name}")
        print("=" * 80)

        loader = get_reliability_case_loader(case_name)
        logits, labels = get_logits_labels(conf_hist_model, loader, device=DEVICE)

        if method == "pseudocal":
            if case_cfg["type"] == "imagenet_c":
                fitted_T = get_fitted_temperature_for_method_seed(
                    method=method,
                    seed=CONF_HIST_REFERENCE_SEED,
                    corruption=case_cfg["corruption"],
                    severity=case_cfg["severity"],
                )
            else:
                fitted_T = 1.0
        else:
            fitted_T = get_fitted_temperature_for_method_seed(
                method=method,
                seed=CONF_HIST_REFERENCE_SEED,
            )

        scaled_logits = apply_temperature(logits, fitted_T)

        figure_name = f"confidence_hist_{method}_{case_name}.png"
        figure_path = CONF_HIST_OUTPUT_DIR / figure_name

        conf_df = plot_confidence_histogram_from_logits(
            logits=scaled_logits,
            labels=labels,
            title=f"Confidence Histogram — {method} — {case_name}",
            save_path=figure_path,
            bins=15,
        )

        conf_csv_path = CONF_HIST_OUTPUT_DIR / f"confidence_hist_{method}_{case_name}.csv"
        save_dataframe(conf_df, conf_csv_path)

        record = {
            "method": method,
            "case": case_name,
            "seed": CONF_HIST_REFERENCE_SEED,
            "temperature": fitted_T,
            "figure_path": str(figure_path),
            "csv_path": str(conf_csv_path),
        }

        if case_cfg["type"] == "imagenet_c":
            record["corruption"] = case_cfg["corruption"]
            record["severity"] = case_cfg["severity"]

        confidence_hist_records.append(record)

confidence_hist_summary_df = pd.DataFrame(confidence_hist_records)
confidence_hist_summary_csv_path = CONF_HIST_OUTPUT_DIR / "confidence_hist_summary.csv"
save_dataframe(confidence_hist_summary_df, confidence_hist_summary_csv_path)

confidence_hist_summary_json = {
    "reference_seed": CONF_HIST_REFERENCE_SEED,
    "methods": CONF_HIST_METHODS,
    "cases": CONF_HIST_CASES,
    "summary_csv": str(confidence_hist_summary_csv_path),
}

confidence_hist_summary_json_path = CONF_HIST_OUTPUT_DIR / "confidence_hist_summary.json"
save_json(confidence_hist_summary_json, confidence_hist_summary_json_path)

print("Saved confidence histogram summary CSV to:")
print(confidence_hist_summary_csv_path)

print("Saved confidence histogram summary JSON to:")
print(confidence_hist_summary_json_path)

display(confidence_hist_summary_df)

## Section 17 — Leakage and fairness audit table

In [ ]:
This section assembles the official leakage and fairness audit table for the rerun protocol.

The purpose of this section is to make the fitting information, evaluation conditions, and possible leakage risks explicit for each method.

The audit table records:

- regime,
- fitting data source,
- whether labels were used,
- whether target data was used,
- whether target labels were used,
- whether synthetic perturbations were used,
- whether benchmark selection was used,
- whether fit and evaluation data were disjoint,
- leakage-risk verdict,
- protocol status.

This table is intended to directly address the protocol concerns raised in supervisor feedback.

audit_df = pd.DataFrame(audit_rows)

# Ensure official column order
audit_df = audit_df[AUDIT_COLUMNS]

print("Audit table shape:", audit_df.shape)
display(audit_df)

AUDIT_OUTPUT_DIR = Path(PATHS["results_root"]) / "audit"
AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

audit_csv_path = AUDIT_OUTPUT_DIR / "leakage_fairness_audit.csv"
save_dataframe(audit_df, audit_csv_path)

print("Saved leakage and fairness audit CSV to:")
print(audit_csv_path)

audit_status_counts = audit_df["status"].value_counts(dropna=False).reset_index()
audit_status_counts.columns = ["status", "count"]

audit_risk_counts = audit_df["leakage_risk"].value_counts(dropna=False).reset_index()
audit_risk_counts.columns = ["leakage_risk", "count"]

print("Audit status counts:")
display(audit_status_counts)

print("Audit leakage-risk counts:")
display(audit_risk_counts)

audit_summary = {
    "num_audit_rows": int(len(audit_df)),
    "methods_audited": sorted(audit_df["method"].dropna().unique().tolist()),
    "all_statuses": sorted(audit_df["status"].dropna().unique().tolist()),
    "all_leakage_risks": sorted(audit_df["leakage_risk"].dropna().unique().tolist()),
    "main_findings": [
        "Baseline uses no fitting data and has no leakage risk.",
        "TS uses only clean labeled source calibration data.",
        "TS-P uses only source calibration data under one fixed proxy perturbation transform.",
        "PseudoCal uses unlabeled target-fit subsets only, with disjoint target-eval subsets for final evaluation.",
        "MM-NLL-TS uses only labeled source calibration data under fixed perturbation views and a frozen minimax objective.",
        "Regime B fairness was repaired by evaluating the baseline on the same disjoint target-eval subsets used for PseudoCal."
    ],
    "overall_protocol_verdict": (
        "Under the frozen rerun notebook, the methods are separated by regime, "
        "recomputed from scratch, and audited for information usage and fit/eval separation."
    ),
}

audit_summary_json_path = AUDIT_OUTPUT_DIR / "audit_summary.json"
save_json(audit_summary, audit_summary_json_path)

print("Saved audit summary JSON to:")
print(audit_summary_json_path)

audit_compact_df = audit_df[[
    "method",
    "regime",
    "fitting_data_source",
    "labels_used",
    "target_data_used",
    "target_labels_used",
    "fit_eval_disjoint",
    "leakage_risk",
    "status",
]]

display(audit_compact_df)

audit_compact_csv_path = AUDIT_OUTPUT_DIR / "leakage_fairness_audit_compact.csv"
save_dataframe(audit_compact_df, audit_compact_csv_path)

print("Saved compact audit CSV to:")
print(audit_compact_csv_path)

## Final summary and export bundle

In [ ]:
This section assembles the final official outputs of the frozen rerun notebook.

The goals of this section are:

- collect the official tables, figures, summaries, and audit files,
- save one final manifest of the output bundle,
- write a careful regime-specific summary of findings,
- avoid any mixed cross-regime “best overall” claim.

The outputs listed here are the official results of this notebook and should be used for the updated report.

final_regime_summary = {
    "protocol_note": (
        "All official results in this project are taken from the frozen rerun notebook. "
        "Methods are separated by regime and recomputed from scratch under a shared protocol."
    ),
    "regime_a_summary": (
        "Regime A compares baseline, TS, and TS-P under clean-source post-hoc calibration. "
        "These methods use only source-side labeled calibration information."
    ),
    "regime_b_summary": (
        "Regime B compares baseline and PseudoCal under a target-unlabeled protocol. "
        "PseudoCal fitting uses only unlabeled target-fit subsets, while final evaluation is reported "
        "only on disjoint target-eval subsets. Baseline was rerun on the same target-eval subsets for fairness."
    ),
    "regime_c_summary": (
        "Regime C compares baseline and MM-NLL-TS under source-side robust calibration. "
        "MM-NLL-TS uses only labeled source calibration data together with fixed perturbation views "
        "and a frozen worst-case NLL objective."
    ),
    "cross_regime_caution": (
        "Results should be interpreted within regime. "
        "No single mixed cross-regime leaderboard should be used as strict scientific evidence."
    ),
}

FINAL_OUTPUT_DIR = Path(PATHS["results_root"]) / "final_exports"
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_manifest = {
    "results_root": str(PATHS["results_root"]),
    "official_protocol": {
        "checkpoint_rule": PROTOCOL["checkpoint_rule"],
        "seeds": PROTOCOL["seeds"],
        "corruptions": PROTOCOL["corruptions"],
        "severities": PROTOCOL["severities"],
        "metrics": PROTOCOL["metrics"],
    },
    "regime_tables": {
        "regime_a_clean_val": str(regime_a_clean_table_path),
        "regime_a_imagenetc_overall": str(regime_a_imagenetc_table_path),
        "regime_b_clean_val": str(regime_b_clean_table_path),
        "regime_b_target_eval": str(regime_b_targeteval_table_path),
        "regime_c_clean_val": str(regime_c_clean_table_path),
        "regime_c_imagenetc_overall": str(regime_c_imagenetc_table_path),
    },
    "aggregated_method_results": {
        "baseline": str(baseline_agg_csv_path),
        "ts": str(ts_agg_csv_path),
        "tsp": str(tsp_agg_csv_path),
        "pseudocal": str(pseudocal_agg_csv_path),
        "mmnll_ts": str(mmnll_agg_csv_path),
        "baseline_regime_b_aligned": str(baseline_b_targeteval_agg_csv_path),
    },
    "fit_summaries": {
        "ts": str(ts_fit_csv_path),
        "tsp": str(tsp_fit_csv_path),
        "pseudocal": str(pseudocal_fit_csv_path),
        "mmnll_ts": str(mmnll_fit_csv_path),
    },
    "key_figures": {
        "regime_a_ece15_vs_severity": str(regime_a_ece_path),
        "regime_a_nll_vs_severity": str(regime_a_nll_path),
        "regime_b_ece15_vs_severity": str(regime_b_ece_path),
        "regime_b_nll_vs_severity": str(regime_b_nll_path),
        "regime_c_ece15_vs_severity": str(regime_c_ece_path),
        "regime_c_nll_vs_severity": str(regime_c_nll_path),
    },
    "diagnostics": {
        "reliability_summary": str(reliability_summary_csv_path),
        "confidence_hist_summary": str(confidence_hist_summary_csv_path),
    },
    "audit": {
        "full_audit_csv": str(audit_csv_path),
        "compact_audit_csv": str(audit_compact_csv_path),
        "audit_summary_json": str(audit_summary_json_path),
    },
    "written_summary": final_regime_summary,
}

final_manifest_json_path = FINAL_OUTPUT_DIR / "final_manifest.json"
save_json(final_manifest, final_manifest_json_path)

print("Saved final manifest to:")
print(final_manifest_json_path)


# Save regime-aware text summary

final_summary_text = f"""
FINAL REGIME-AWARE SUMMARY

Protocol note:
{final_regime_summary['protocol_note']}

Regime A:
{final_regime_summary['regime_a_summary']}

Regime B:
{final_regime_summary['regime_b_summary']}

Regime C:
{final_regime_summary['regime_c_summary']}

Cross-regime caution:
{final_regime_summary['cross_regime_caution']}
""".strip()

final_summary_txt_path = FINAL_OUTPUT_DIR / "final_summary.txt"
with open(final_summary_txt_path, "w", encoding="utf-8") as f:
    f.write(final_summary_text)

print("Saved final summary text to:")
print(final_summary_txt_path)
print()
print(final_summary_text)


# Final output checklist display

final_checklist_df = pd.DataFrame([
    {"category": "Regime table", "name": "Regime A clean val", "path": str(regime_a_clean_table_path)},
    {"category": "Regime table", "name": "Regime A ImageNet-C", "path": str(regime_a_imagenetc_table_path)},
    {"category": "Regime table", "name": "Regime B clean val", "path": str(regime_b_clean_table_path)},
    {"category": "Regime table", "name": "Regime B target-eval", "path": str(regime_b_targeteval_table_path)},
    {"category": "Regime table", "name": "Regime C clean val", "path": str(regime_c_clean_table_path)},
    {"category": "Regime table", "name": "Regime C ImageNet-C", "path": str(regime_c_imagenetc_table_path)},
    {"category": "Aggregate", "name": "Baseline aggregated", "path": str(baseline_agg_csv_path)},
    {"category": "Aggregate", "name": "TS aggregated", "path": str(ts_agg_csv_path)},
    {"category": "Aggregate", "name": "TS-P aggregated", "path": str(tsp_agg_csv_path)},
    {"category": "Aggregate", "name": "PseudoCal aggregated", "path": str(pseudocal_agg_csv_path)},
    {"category": "Aggregate", "name": "MM-NLL aggregated", "path": str(mmnll_agg_csv_path)},
    {"category": "Audit", "name": "Leakage/fairness audit", "path": str(audit_csv_path)},
    {"category": "Manifest", "name": "Final manifest", "path": str(final_manifest_json_path)},
    {"category": "Summary", "name": "Final summary text", "path": str(final_summary_txt_path)},
])

display(final_checklist_df)